# 02 — Translation Pipeline Overview

Descriptive statistics on the full pipeline output: 12 services (4 baseline + 8 LLM), 880 languages, 4 prompt variants.
This notebook establishes the empirical baseline before the disagreement analysis:

- How many languages have translations from each service, and how does coverage distribute across language families?
- Which services fail on which families, and why? What do the error logs reveal about failure modes?
- How consistent are LLM translations in length across prompt variants and language families?
- What script and directionality anomalies arise, and how are they curated?
- What data-quality signals (missing rationales, mixed-script outputs, script disagreements) are exported for downstream analysis?


In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df

DATA_DIR = get_data_directory_path()
TERMS = ["Digital Humanities"]
VARIANTS = ["minimal", "fluent_speaker", "github_searcher", "judge"]

SERVICE_COLS = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}

print(f"Data directory: {DATA_DIR}")

Retrieving translation pipeline data directory path...

Data directory: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets


In [2]:
TARGET_TERMS = ["Digital Humanities"]

def load_all_variants(data_dir, term, variants=VARIANTS):
    """Merge per-service files for all variants into one concatenated DataFrame."""
    dfs = []
    term_slug = term.lower().replace(" ", "_")
    for variant in variants:
        df = load_variant_df(data_dir, term_slug, variant)
        if df is not None:
            df["term_source_query"] = term
            dfs.append(df)
        else:
            print(f"  ⚠ No data for variant: {variant}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

all_dfs = {term: load_all_variants(DATA_DIR, term) for term in TARGET_TERMS}
for term, df in all_dfs.items():
    print(f"{term}: {len(df)} rows across {df['prompt_variant'].nunique()} variants")

all_terms_df = all_dfs[TARGET_TERMS[0]]

# Total translations before any exclusions
# Use minimal variant so baseline columns are not counted 4x (identical across variants)
_ref        = all_terms_df[all_terms_df["prompt_variant"] == VARIANTS[0]]
_trans_cols = [c for c in all_terms_df.columns if c.endswith("_translated_term")]
_n_lang     = _ref["language_code"].nunique()
_n_svc      = len(_trans_cols)
_n_present  = int(sum(_ref[c].notna().sum() for c in _trans_cols))
_n_possible = _n_lang * _n_svc
print(f"Total translations before exclusions: {_n_present:,} / {_n_possible:,} possible ",f"({_n_lang} languages × {_n_svc} services, {_n_present / _n_possible * 100:.1f}% filled)")

Digital Humanities: 3520 rows across 4 variants
Total translations before exclusions: 7,232 / 10,560 possible  (880 languages × 12 services, 68.5% filled)


## 2.1 Pipeline Coverage

### Coverage by Service

How many of the 880 languages have a translation from each service, and which services still show up for the hardest-to-translate languages?

In [3]:
BASELINE_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
LLM_SERVICES = {
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}

term = TARGET_TERMS[0]
df = all_dfs[term]

ref = df[df["prompt_variant"] == VARIANTS[0]]
total = ref["language_code"].nunique()

baseline_rows = []
for service, col in BASELINE_SERVICES.items():
    n = ref[col].notna().sum() if col in ref.columns else 0
    baseline_rows.append({"service": service, "n": int(n), "total": total})
baseline_cov = pd.DataFrame(baseline_rows)

llm_rows = []
for variant in VARIANTS:
    vdf = df[df["prompt_variant"] == variant]
    v_total = vdf["language_code"].nunique()
    for service, col in LLM_SERVICES.items():
        n = vdf[col].notna().sum() if col in vdf.columns else 0
        llm_rows.append({"service": service, "variant": variant, "n": int(n), "total": v_total})
llm_cov = pd.DataFrame(llm_rows)

color_scale = alt.Color("service:N", scale=alt.Scale(scheme="tableau10"), title="Service")

# Chart 1: baseline coverage
baseline_bars = alt.Chart(baseline_cov).mark_bar().encode(
    y=alt.Y("service:N", sort="-x", title=None),
    x=alt.X("n:Q", scale=alt.Scale(domain=[0, total]), title="languages translated"),
    color=color_scale,
    tooltip=["service:N", alt.Tooltip("n:Q", title="languages"), alt.Tooltip("total:Q", title="total")],
).properties(width=350, height=160, title=f"Baseline service coverage (n={total} languages, prompt-invariant)")

baseline_text = baseline_bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text="n:Q", color=alt.value("black"), opacity=alt.value(1),
)

# Chart 2: LLM coverage by variant
llm_bars = alt.Chart(llm_cov).mark_bar().encode(
    y=alt.Y("variant:N", sort=VARIANTS, title=None),
    x=alt.X("n:Q", title="languages translated"),
    yOffset="service:N",
    color=color_scale,
    tooltip=["service:N", "variant:N", alt.Tooltip("n:Q", title="languages"), alt.Tooltip("total:Q", title="total")],
).properties(width=380, height=280, title="LLM service coverage by prompt variant")

# Chart 3: connected scatter — service lines across variants
scatter_line = alt.Chart(llm_cov).mark_line(opacity=0.35, strokeWidth=1.5).encode(
    x=alt.X("variant:N", sort=VARIANTS, title=None),
    y=alt.Y("n:Q", title="languages translated", scale=alt.Scale(zero=False)),
    color=color_scale,
    detail="service:N",
)
scatter_pts = alt.Chart(llm_cov).mark_point(filled=True, size=80).encode(
    x=alt.X("variant:N", sort=VARIANTS, title=None),
    y=alt.Y("n:Q", title="languages translated", scale=alt.Scale(zero=False)),
    color=color_scale,
    tooltip=["service:N", "variant:N", alt.Tooltip("n:Q", title="languages")],
)
scatter = (scatter_line + scatter_pts).properties(
    width=380, height=220,
    title="Coverage per service across variants (flat line = variant has no effect)",
)

((baseline_bars + baseline_text) & llm_bars & scatter).display()

# Chart 4: which services show up for the hardest languages (covered by ≤5 services total).
all_svc_cols = {**BASELINE_SERVICES, **LLM_SERVICES}
all_svc_cols = {s: c for s, c in all_svc_cols.items() if c in ref.columns}
cov_mat = pd.DataFrame(
    {s: ref[c].notna().values for s, c in all_svc_cols.items()},
    index=ref["language_code"].values,
)
cov_mat["n_total"] = cov_mat[list(all_svc_cols.keys())].sum(axis=1)
sparse = cov_mat[cov_mat["n_total"] <= cov_mat.n_total.describe()['25%']]

sparse_rows = []
for service, col in all_svc_cols.items():
    sparse_rows.append({
        "service": service,
        "n": int(cov_mat.loc[sparse.index, service].sum()),
        "total": len(sparse),
    })
sparse_df = pd.DataFrame(sparse_rows)

sparse_bar = alt.Chart(sparse_df).mark_bar().encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("n", order="descending"), title=None),
    x=alt.X("n:Q", scale=alt.Scale(domain=[0, len(sparse)]),
            title=f"languages translated (of {len(sparse)} covered by {cov_mat.n_total.describe()['25%']} services)"),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N", alt.Tooltip("n:Q", title="hardest languages covered"),
             alt.Tooltip("total:Q", title="total hardest")],
).properties(width=380, height=220, title="Which services cover the hardest languages?")

sparse_text = sparse_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text="n:Q", color=alt.value("black"))

(sparse_bar + sparse_text)

alt.VConcatChart(...)

alt.LayerChart(...)

### Coverage by Language Family

How does each service's coverage distribute across language families? A cell is green if the service translated at least one prompt variant for that family; white/empty means no coverage. LLM services aggregate across all 4 prompt variants (any variant = covered); baseline services run once per language.

In [4]:
HEATMAP_SERVICES = {
    "Wikipedia":        "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT":          "enmt_translated_term",
    "Lingvanex":        "lingvanex_translated_term",
    "OpenAI":           "openai_translated_term",
    "Claude":           "claude_translated_term",
    "Gemini":           "gemini_translated_term",
    "DeepSeek":         "deepseek_translated_term",
    "Llama":            "llama_translated_term",
    "Gemma":            "gemma_translated_term",
    "Qwen":             "qwen_translated_term",
    "Mistral":          "mistral_translated_term",
}
service_order = list(HEATMAP_SERVICES.keys())

term = TARGET_TERMS[0]
all_variants = all_dfs[term].copy()
all_variants["language_family"] = all_variants["language_code"].apply(get_language_family)

# Max coverage across all prompt variants: a language counts as covered by a
# service if any variant produced a valid translation for it.
avail_cols = {svc: col for svc, col in HEATMAP_SERVICES.items() if col in all_variants.columns}
baseline = (
    all_variants
    .assign(**{col: all_variants[col].notna() for col in avail_cols.values()})
    .groupby("language_code")
    .agg({col: "max" for col in avail_cols.values()} | {"language_family": "first"})
    .reset_index()
)

other_langs = baseline[baseline["language_family"] == "Other"]["language_code"].tolist()
if other_langs:
    print(f"⚠ Ungrouped languages: {other_langs}")
else:
    print("✓ All languages assigned to a family")

family_counts = baseline["language_family"].value_counts().to_dict()

rows = []
for family, n_fam in sorted(family_counts.items(), key=lambda x: -x[1]):
    fam_df = baseline[baseline["language_family"] == family]
    label = f"{family} ({n_fam})"
    for service, col in HEATMAP_SERVICES.items():
        n_translated = int(fam_df[col].sum()) if col in fam_df.columns else 0
        pct = round(n_translated / n_fam * 100, 1) if n_fam else 0.0
        rows.append({"Family": label, "Service": service, "Coverage": pct, "N": n_translated, "Family_size": n_fam})
coverage_long = pd.DataFrame(rows)

family_order = [f"{f} ({family_counts[f]})" for f in sorted(family_counts.keys(), key=lambda k: -family_counts[k])]
chart = alt.Chart(coverage_long).mark_rect().encode(
    x=alt.X("Service:N", sort=service_order, title="Translation Service"),
    y=alt.Y("Family:N", sort=family_order, title="Language Family"),
    color=alt.Color("Coverage:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, 100]),
        legend=alt.Legend(title="% coverage"),
    ),
    tooltip=[
        "Family", "Service",
        alt.Tooltip("Coverage:Q", format=".1f", title="% coverage"),
        alt.Tooltip("N:Q", title="languages translated"),
        alt.Tooltip("Family_size:Q", title="family size"),
    ],
).properties(
    title=alt.Title(
        "Languages Translated by Service and Language Family",
        subtitle="LLM services: coverage across all prompt variants (any variant = covered). Baseline services run once.",
    ),
    width=520, height=420,
)
text = chart.mark_text(baseline="middle", fontSize=8).encode(
    text=alt.Text("Coverage:Q", format=".0f"),
    color=alt.condition(alt.datum.Coverage > 60, alt.value("white"), alt.value("black")),
)
(chart + text)

Retrieving translation pipeline data directory path...

✓ All languages assigned to a family


alt.LayerChart(...)

### Languages with Zero Coverage

These are the most interesting cases for the paper — languages where the pipeline produced nothing at all. Are they low-resource languages? Script-diverse languages? Languages where 'Digital Humanities' genuinely has no circulation?

In [5]:
term = TARGET_TERMS[0]
df_all = all_dfs[term].copy()
df_all["language_family"] = df_all["language_code"].apply(get_language_family)

all_svc_cols = {**BASELINE_SERVICES, **LLM_SERVICES}
svc_cols = [c for c in all_svc_cols.values() if c in df_all.columns]

# A service "covers" a language if it succeeded in at least one prompt variant
coverage = df_all.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any()).reset_index()
lang_meta = df_all[["language_code", "language_name", "language_family"]].drop_duplicates("language_code")
baseline = coverage.merge(lang_meta, on="language_code")
baseline["n_services"] = baseline[svc_cols].sum(axis=1)

TIERS = {"0 — none": (0,0), "1–2 — sparse": (1,2), "3–5 — partial": (3,5), "6–9 — rich": (6,9)}
def tier(n):
    for label, (lo, hi) in TIERS.items():
        if lo <= n <= hi: return label
    return "other"
baseline["tier"] = baseline["n_services"].apply(tier)

tier_order = list(TIERS.keys())
hist = alt.Chart(baseline).mark_bar().encode(
    x=alt.X("n_services:O", title="Services that translated this language (any prompt)", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Number of languages"),
    color=alt.Color("tier:N", sort=tier_order,
        scale=alt.Scale(domain=tier_order, range=["#d32f2f","#ff9800","#1976d2","#388e3c"]),
        title="Coverage tier"),
    tooltip=["n_services:O", "count():Q", "tier:N"],
).properties(width=360, height=240, title="Service coverage depth per language (best across all prompts)")

low_cov = baseline[baseline["n_services"] <= 3][
    ["language_code","language_name","language_family","n_services","tier"]
].sort_values(["n_services","language_family"]).reset_index(drop=True)

print(f"Zero coverage: {(baseline['n_services']==0).sum()} | Sparse (1-3): {((baseline['n_services']>=1)&(baseline['n_services']<=3)).sum()}. Only sparse languages: {low_cov['language_name'].tolist()}")

hist

Zero coverage: 0 | Sparse (1-3): 0. Only sparse languages: []


alt.Chart(...)

### Error Log Analysis

What do the error logs tell us about *why* translations failed?

The first chart shows raw error counts per service by HTTP status code. Baseline services (EasyNMT, Google Translate, Lingvanex, Wikipedia) have no `error_message` column — their errors are pure HTTP failures (EasyNMT 404 indicates unsupported language pair; whereas GT/Lingvanex 400/500 indicates quota or API errors).

The second chart classifies LLM error messages into nine categories (priority-ordered — first match wins):

- **`max_tokens`** — context length exceeded before a parseable term could be extracted (Gemini, Claude)
- **`api_error`** — upstream 500 from the provider
- **`empty_translation`** — model returned a valid response but with an empty `translated_term` field (Llama 25, Qwen 8)
- **`ollama_timeout`** — Ollama returned only a `total_duration` field with no content, indicating the model timed out before generation completed (Gemma 8, Qwen 3)
- **`extinct_ancient`** — model explicitly cites the language being extinct, ancient, or undeciphered
- **`knowledge_gap`** — model cites insufficient data or capability for this language (broader than extinct_ancient)
- **`generic_refusal`** — vague refusal with no stated reason (includes OpenAI's "I'm sorry, but I can't provide a translation for..." pattern)
- **`repetition_loop`** — model produced a structurally valid JSON object but the `translated_term` is a hallucinated repetition of a short syllable or character sequence; detected by unique-character density < 15% of term length (DeepSeek, Gemma, Llama, Mistral, OpenAI, Qwen)
- **`parse_format`** — malformed or non-JSON response that cleared all earlier checks


In [6]:
SERVICE_NAMES = {
    "claude":    "Claude",   "deepseek": "DeepSeek", "enmt":  "EasyNMT",
    "gemini":    "Gemini",   "gemma":    "Gemma",    "gt":    "Google Translate",
    "lingvanex": "Lingvanex","llama":    "Llama",    "mistral": "Mistral",
    "openai":    "OpenAI",   "qwen":     "Qwen",     "wikipedia": "Wikipedia",
}

error_dir = os.path.join(DATA_DIR, "error_logs")
error_rows = []

if os.path.exists(error_dir):
    for fname in sorted(os.listdir(error_dir)):
        if not fname.endswith(".csv"): continue
        key = fname.replace("_translation_errors.csv", "")
        name = SERVICE_NAMES.get(key, key)
        try:
            edf = read_csv_file(os.path.join(error_dir, fname))
            if "status_code" in edf.columns:
                for code, cnt in edf["status_code"].value_counts().items():
                    error_rows.append({"service": name, "status": str(code), "count": int(cnt)})
            else:
                error_rows.append({"service": name, "status": "unknown", "count": len(edf)})
        except Exception as e:
            print(f"Could not read {fname}: {e}")

error_df = pd.DataFrame(error_rows)

if error_df.empty:
    print("No error rows found — skipping error charts.")
else:
    totals = error_df.groupby("service")["count"].sum().reset_index().sort_values("count", ascending=False)
    svc_order = totals["service"].tolist()

    total_bar = alt.Chart(totals).mark_bar().encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("count:Q", title="total errors logged"),
        color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
        tooltip=["service:N", "count:Q"],
    ).properties(width=320, height=220, title="Total errors per service")

    total_text = total_bar.mark_text(align="left", dx=4, fontSize=9).encode(
        text="count:Q", color=alt.value("black"))

    status_bar = alt.Chart(error_df).mark_bar().encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("count:Q", title="error count"),
        color=alt.Color("status:N", title="HTTP status", scale=alt.Scale(scheme="set2")),
        order=alt.Order("count:Q", sort="descending"),
        tooltip=["service:N", "status:N", "count:Q"],
    ).properties(width=320, height=220, title="Error breakdown by status code")

    display((total_bar + total_text) | status_bar)

alt.HConcatChart(...)

In [7]:
PARSE_PREFIX = "Could not parse translation response: "
LLM_ERROR_FILES = {
    "Claude":   "claude_translation_errors.csv",
    "OpenAI":   "openai_translation_errors.csv",
    "Gemini":   "gemini_translation_errors.csv",
    "DeepSeek": "deepseek_translation_errors.csv",
    "Llama":    "llama_translation_errors.csv",
    "Gemma":    "gemma_translation_errors.csv",
    "Qwen":     "qwen_translation_errors.csv",
    "Mistral":  "mistral_translation_errors.csv",
}

CATEGORIES = [
    ("max_tokens",        ["MAX_TOKENS"]),
    ("api_error",         ["INTERNAL", "An internal error has occurred",
                           "read operation timed out"]),
    ("empty_translation", ["Model returned empty translated_term"]),
    ("ollama_timeout",    ["'total_duration'"]),
    ("extinct_ancient",   ["extinct", "ancient", "undeciphered", "no native speakers",
                           "historical language", "limited corpus", "Minoan", "funerary",
                           "no longer spoken"]),
    ("knowledge_gap",     ["limited resources", "limited documentation", "limited data",
                           "limited information", "limited available",
                           "not have the ability", "not able to provide",
                           "don't have the capability", "do not have the capacity",
                           "not equipped", "knowledge cutoff", "training data",
                           "not within my capabilities", "cannot provide",
                           "unable to provide", "not sufficiently documented",
                           "not thoroughly documented", "not widely documented",
                           "under-documented", "lesser-known", "lesser-documented",
                           "less commonly", "less widely", "no comprehensive",
                           "no data on", "not have data", "do not have specific",
                           "don't have specific",
                           "does not have a direct equivalent",
                           "does not directly translate",
                           "no available data", "no available translation",
                           "no known translation",
                           "couldn't find", "could not find"]),
    ("generic_refusal",   ["I'm sorry, I can't", "I'm sorry, I cannot",
                           "I'm sorry, but",
                           "I must respectfully decline",
                           "I can't assist", "I can't comply", "I can't do that",
                           "I cannot perform", "No rationale provided",
                           "cannot fulfill", "can't fulfill", "unable to fulfill",
                           "unable to translate", "not_available"]),
]

import re as _re

def _is_repetition_loop(raw_msg: str) -> bool:
    """True if the error wraps a translated_term that is a hallucination repetition loop.

    Strips the pipeline parse-prefix and any ```json marker, then extracts the
    translated_term value. Two signals are checked:
      1. Unique-character density < 15% (e.g. "Mbaŋaŋaŋ..." has only 4 distinct chars).
      2. Any 2–10 char substring starting at one of the first 5 positions repeats ≥5 times.
    """
    text = raw_msg.replace(PARSE_PREFIX, "").strip()
    text = _re.sub(r"^```json\s*", "", text).strip()
    m = _re.search(r'"translated_term"\s*:\s*"(.{15,})', text)
    if not m:
        return False
    term = m.group(1).rstrip('"} \n\r\t')
    if len(term) < 20:
        return False
    if len(set(term)) / len(term) < 0.15:
        return True
    for start in range(min(5, len(term))):
        for chunk_len in range(2, min(len(term) // 4 + 1, 12)):
            chunk = term[start : start + chunk_len]
            if chunk and term.count(chunk) >= 5:
                return True
    return False

def classify_error(msg):
    if not isinstance(msg, str):
        return "unknown"
    text = msg.replace(PARSE_PREFIX, "").strip()
    # Repetition-loop check before parse_format: the JSON may be structurally valid
    # but the translated_term is a hallucination loop.
    if _is_repetition_loop(msg):
        return "repetition_loop"
    if text.startswith("{") or '"translated_term"' in text:
        return "parse_format"
    for cat, keywords in CATEGORIES:
        if any(kw.lower() in text.lower() for kw in keywords):
            return cat
    return "other"

lang_names = (
    all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]]
    .drop_duplicates("language_code")
    .set_index("language_code")["language_name"]
    .to_dict()
)

llm_frames = []
for service, fname in LLM_ERROR_FILES.items():
    path = os.path.join(error_dir, fname)
    if not os.path.exists(path):
        continue
    edf = read_csv_file(path)
    edf["service"] = service
    edf["category"] = edf["error_message"].apply(classify_error)
    edf["clean_msg"] = (
        edf["error_message"].str.replace(PARSE_PREFIX, "", regex=False).str.strip()
    )
    edf["language_name"] = edf["language_code"].map(lang_names).fillna(edf["language_code"])
    llm_frames.append(edf)

llm_err = pd.concat(llm_frames, ignore_index=True)

print(f"Total LLM errors: {len(llm_err)} across {llm_err['language_code'].nunique()} unique languages\n")
print(f"Total unique error messages: {llm_err['error_message'].nunique()}\n")
print("Error messages that appeared more than once:")
grouped_llm_err = llm_err.error_message.value_counts().reset_index()
grouped_llm_err[grouped_llm_err['count'] > 1]


Total LLM errors: 794 across 378 unique languages

Total unique error messages: 705

Error messages that appeared more than once:


,error_message,count
0,"500 INTERNAL. {'error': {'code': 500, 'message...",23
1,Model returned empty translated_term,19
2,done=False — model hit token limit mid-generation,10
3,'NoneType' object has no attribute 'strip',4
4,Could not parse translation response: I don't ...,3
5,Could not parse translation response: I can't ...,3
6,Could not parse translation response: I don't ...,3
7,Could not parse translation response: ```json\...,2
8,Could not parse translation response: I'm sorr...,2
9,Could not parse translation response: I'm sorr...,2


In [8]:
print(llm_err.groupby(["service", "category"]).size().unstack(fill_value=0).to_string())

# Stacked bar: category breakdown per service
cat_order = (
    [c for c, _ in CATEGORIES]
    + ["repetition_loop", "parse_format", "other", "unknown"]
)
cat_counts = llm_err.groupby(["service", "category"]).size().reset_index(name="n")

err_chart = alt.Chart(cat_counts).mark_bar().encode(
    y=alt.Y("service:N", title=None),
    x=alt.X("n:Q", title="errors"),
    color=alt.Color("category:N", sort=cat_order,
                    scale=alt.Scale(scheme="tableau10"), title="Category"),
    order=alt.Order("n:Q", sort="descending"),
    tooltip=["service:N", "category:N", "n:Q"],
).properties(width=420, height=180, title="LLM error categories per service")
display(err_chart)

# Drill-down: languages in knowledge_gap, extinct_ancient, and repetition_loop
for cat in ("knowledge_gap", "extinct_ancient", "repetition_loop"):
    sub = llm_err[llm_err["category"] == cat][
        ["service", "language_code", "language_name", "clean_msg"]
    ].copy()
    if sub.empty:
        continue
    print(f"\n── {cat}  ({len(sub)} errors, {sub['language_code'].nunique()} languages) ──")

    # Languages flagged by 2+ services — the pipeline's true blind spots
    multi_svc = (
        sub.groupby(["language_code", "language_name"])
        .agg(
            n_services=("service", "nunique"),
            services=("service", lambda x: sorted(x.unique())),
        )
        .reset_index()
        .query("n_services >= 2")
        .sort_values("n_services", ascending=False)
    )
    if not multi_svc.empty:
        print(f"  Flagged by ≥2 services ({len(multi_svc)} languages):")
        for _, r in multi_svc.iterrows():
            print(f"    {r['language_code']} ({r['language_name']}): {r['services']}")

    # Sample messages (2 per service)
    print(f"\n  Sample messages:")
    for svc, grp in sub.groupby("service"):
        print(f"  [{svc}]")
        for _, row in grp.head(2).iterrows():
            print(f"    {row['language_code']} ({row['language_name']}): {row['clean_msg'][:140]}")


category  api_error  empty_translation  extinct_ancient  generic_refusal  knowledge_gap  max_tokens  other  parse_format  repetition_loop  unknown
service                                                                                                                                           
Claude            0                  0                0                0              0           3      0             0                0        0
DeepSeek          0                  0                0                0              0           0      0             0               32        0
Gemini           24                  0                1                4             10          65      6             1                0        0
Gemma             0                  0                0                0              0           0     10             0               44        0
Llama             0                 10                3                8             54           0     25            

alt.Chart(...)


── knowledge_gap  (188 errors, 141 languages) ──
  Flagged by ≥2 services (19 languages):
    akz (Alabama): ['Llama', 'OpenAI']
    mde (Maba): ['Llama', 'OpenAI']
    xlc (Lycian): ['Gemini', 'OpenAI']
    xcr (Carian): ['Gemini', 'OpenAI']
    unx (Munda): ['Llama', 'OpenAI']
    tht (Tahltan): ['Llama', 'OpenAI']
    oka (Okanagan): ['Llama', 'Mistral']
    mgy (Mbunga): ['Gemini', 'OpenAI']
    mdt (Mbere): ['Llama', 'OpenAI']
    kxv (Kuvi): ['Llama', 'OpenAI']
    aro (Araona): ['Gemini', 'OpenAI']
    hur (Halkomelem): ['Gemini', 'OpenAI']
    gld (Nanai): ['Llama', 'OpenAI']
    gjk (Kachi Koli): ['Llama', 'OpenAI']
    fud (East Futuna): ['Mistral', 'OpenAI']
    car (Carib): ['Gemini', 'OpenAI']
    bqv (Koro Wachi): ['Gemini', 'OpenAI']
    bmq (Bomu): ['Gemini', 'Llama']
    xmn (Manichaean Middle Persian): ['Llama', 'OpenAI']

  Sample messages:
  [Gemini]
    cay (Cayuga): I am unable to provide a translation of "Digital Humanities" into Cayuga. Here's why:

* **Lack of

#### Repetition-Loop Language Profile

Sixty-nine errors across 50 languages and 6 services (DeepSeek, Gemma, Llama, Mistral, OpenAI, Qwen) show a distinctive failure pattern: the model produces syntactically valid JSON but the `translated_term` value is a short syllable or character sequence repeated until the field is hundreds of characters long (e.g. `"Mbaŋaŋaŋaŋaŋ..."`, `"ᑎᒥᔅᑭᓂᐦᐄᑭᓂᐦᐄ..."`). This is a model-side generation artifact—not a prompt or parsing problem—and is detected by the `_is_repetition_loop()` check applied before all other classifiers.

Two structural drivers appear:
- **Low-resource phonology**: languages with complex tone or nasal-vowel sequences (Bantoid, Nilo-Saharan, Algonquian syllabics) where the model "locks on" to a recurrent phoneme.
- **Rare or extinct scripts**: cuneiform (Akkadian, Elamite, Hittite), Sogdian, Meroitic, Linear A — the model has almost no training data in the target script and halluccinates token-repetition loops.

Languages flagged by 2+ services are the pipeline's hardest cases — translation is unreliable regardless of service.


In [9]:
# Repetition-loop language profile
rep = llm_err[llm_err["category"] == "repetition_loop"].copy()

print(f"Repetition-loop errors: {len(rep)} across {rep['language_code'].nunique()} languages")

# Multi-service languages (hardest cases)
multi = (
	rep.groupby(["language_code", "language_name"])
	.agg(n_services=("service", "nunique"), services=("service", lambda x: sorted(x.unique())))
	.reset_index()
	.query("n_services >= 2")
	.sort_values("n_services", ascending=False)
)
print(f"Languages flagged by ≥2 services ({len(multi)}):")
for _, r in multi.iterrows():
	print(f"  {r['language_code']} ({r['language_name']}): {r['services']}")

# Cross-check: do any also have an honest refusal from a *different* service?
print("Honest-refusal cross-service overlap (loop + honest refusal from different service):")
for lang, grp in rep.groupby("language_code"):
	rep_svcs = set(grp["service"])
	honest = llm_err[
		(llm_err["language_code"] == lang) &
		(llm_err["category"].isin(["knowledge_gap", "extinct_ancient"])) &
		(~llm_err["service"].isin(rep_svcs))
	]
	if not honest.empty:
		lang_name = grp["language_name"].iloc[0]
		print(f"  {lang} ({lang_name}): loops in {sorted(rep_svcs)}, "
			  f"honest refusal ({sorted(honest['category'].unique())}) from {sorted(honest['service'].unique())}")

print("Per-service repetition-loop count:")
print(rep["service"].value_counts().to_string())

# Family breakdown (unique languages only)
# language_family is not a column in all_dfs — derive it via get_language_family
_fam_map = {
	code: get_language_family(code)
	for code in all_dfs[TARGET_TERMS[0]]["language_code"].unique()
}
rep_unique = rep.drop_duplicates("language_code").copy()
rep_unique["language_family"] = rep_unique["language_code"].map(_fam_map)
print("Family breakdown (unique languages):")
print(rep_unique["language_family"].value_counts().to_string())

Repetition-loop errors: 104 across 85 languages
Languages flagged by ≥2 services (6):
  bqv (Koro Wachi): ['DeepSeek', 'Gemma']
  crl (Northern East Cree): ['Gemma', 'OpenAI']
  kpy (Koryak): ['Gemma', 'Llama']
  pqm (Maliseet-Passamaquoddy): ['DeepSeek', 'Qwen']
  sog (Sogdian): ['DeepSeek', 'Mistral']
  xmr (Meroitic): ['DeepSeek', 'Gemma']
Honest-refusal cross-service overlap (loop + honest refusal from different service):
  abr (Abron): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['Llama']
  akz (Alabama): loops in ['Gemma'], honest refusal (['knowledge_gap']) from ['Llama', 'OpenAI']
  bqv (Koro Wachi): loops in ['DeepSeek', 'Gemma'], honest refusal (['knowledge_gap']) from ['Gemini', 'OpenAI']
  bsc (bsc): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['Llama']
  cch (Atsam): loops in ['Gemma'], honest refusal (['knowledge_gap']) from ['OpenAI']
  dua (Duala): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['Gemini']
  ecy (Eteo

#### Refusal Grammar

OpenAI accounts for 239 of the pipeline's refusals — more than any other service. Unlike Llama (which uses narrative "Unfortunately, I couldn't find..." openings) or Gemini (5 refusals using "I am unable to provide..."), OpenAI's refusals are formulaic: 97.5% of its generic-refusal and knowledge-gap errors begin with one of five opening phrases, with the single template _"I'm sorry, but I can't provide a translation"_ accounting for 83.7% (200/239).

This uniformity has an important implication for the disagreement analysis: OpenAI refusals are structurally indistinguishable from each other, so two services "disagreeing" may in reality be one refusing and one hallucinating — a failure of different kinds, not a genuine dispute about the translation.


In [10]:
# OpenAI refusal grammar analysis
oa = llm_err[llm_err["service"] == "OpenAI"]
refusals = oa[oa["category"].isin(["generic_refusal", "knowledge_gap"])].copy()

print(f"OpenAI refusal errors: {len(refusals)} ({len(refusals[refusals['category']=='generic_refusal'])} generic + {len(refusals[refusals['category']=='knowledge_gap'])} knowledge_gap)")
print()

openings = refusals["clean_msg"].str.split().str[:8].str.join(" ")
top = openings.value_counts()
print("Top opening phrases (first 8 words):")
cumulative = 0
for i, (phrase, n) in enumerate(top.head(8).items()):
    pct = n / len(refusals) * 100
    cumulative += n
    cum_pct = cumulative / len(refusals) * 100
    print(f"  {n:3d} ({pct:5.1f}% | cum {cum_pct:5.1f}%) '{phrase}'")

top5_n = top.head(5).sum()
print(f"\nTop 5 openings cover {top5_n}/{len(refusals)} = {top5_n/len(refusals)*100:.1f}% of OpenAI refusals")

# Compare with other services' refusal patterns
print("\nComparison: refusal openings by service")
for svc in sorted(llm_err["service"].unique()):
    if svc == "OpenAI":
        continue
    svc_ref = llm_err[(llm_err["service"] == svc) & (llm_err["category"].isin(["generic_refusal", "knowledge_gap"]))]
    if len(svc_ref) == 0:
        continue
    print(f"  {svc} ({len(svc_ref)} refusals):")
    svc_openings = svc_ref["clean_msg"].str.split().str[:8].str.join(" ").value_counts()
    for phrase, n in svc_openings.head(3).items():
        print(f"    {n:3d}  '{phrase}'")


OpenAI refusal errors: 370 (251 generic + 119 knowledge_gap)

Top opening phrases (first 8 words):
  239 ( 64.6% | cum  64.6%) 'I'm sorry, but I can't provide a translation'
   77 ( 20.8% | cum  85.4%) 'I'm sorry, but I currently do not have'
   22 (  5.9% | cum  91.4%) 'I'm sorry, but I cannot provide a translation'
   13 (  3.5% | cum  94.9%) 'I'm sorry, but as of my last update,'
    8 (  2.2% | cum  97.0%) 'I'm sorry, but I don't have the capability'
    2 (  0.5% | cum  97.6%) 'I'm sorry, but I can't assist with that'
    2 (  0.5% | cum  98.1%) 'I'm sorry, but it seems there might be'
    2 (  0.5% | cum  98.6%) 'I'm sorry, but I currently don't have the'

Top 5 openings cover 359/370 = 97.0% of OpenAI refusals

Comparison: refusal openings by service
  Gemini (14 refusals):
      4  'I am unable to provide a translation of'
      2  'I cannot provide a translation of "Digital Humanities"'
      1  'I must respectfully decline to provide a translation'
  Llama (62 refusals):
    

## 2.2 Translation Length and Consistency

Two complementary views of translation length:

**Baseline services** — how long are translations from Wikipedia, Google Translate, EasyNMT, and Lingvanex? These run once per language with no prompt variation, so the histogram gives a clean snapshot of absolute length by service.

**LLM consistency** — how consistently do LLMs produce translations of similar length *within* a language family, *across* all 4 prompt variants? The metric is Coefficient of Variation (CV = std / mean word count): lower CV means the model converges on a consistent length for that family; higher CV means it is improvising differently for each language. Baseline services are excluded here because they have no prompt dimension to compare.

In [11]:
term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()

length_rows = []
for service, col in BASELINE_SERVICES.items():
    if col not in baseline.columns: continue
    for wc in baseline[col].dropna().astype(str).str.split().str.len():
        length_rows.append({"Service": service, "Word Count": int(wc)})
length_df = pd.DataFrame(length_rows)

bars = alt.Chart().mark_bar(opacity=0.85).encode(
    x=alt.X("Word Count:Q", bin=alt.Bin(extent=[1,12], step=1), title="Word count"),
    y=alt.Y("count():Q", title="N languages"),
    color=alt.Color("Service:N", scale=alt.Scale(scheme="tableau10"), legend=None),
    tooltip=["Service", alt.Tooltip("count():Q", title="Count")],
).properties(width=155, height=110)

median_rule = alt.Chart().mark_rule(color="red", strokeDash=[4,2]).encode(
    x=alt.X("median(Word Count):Q"),
    tooltip=[alt.Tooltip("median(Word Count):Q", format=".1f", title="Median")],
)

alt.layer(bars, median_rule, data=length_df).facet(
    facet=alt.Facet("Service:N", sort=list(BASELINE_SERVICES.keys()), title=None),
    columns=2,
    title=f"Translation Word Count Distribution — {term} (red line = median)",
).display()

alt.FacetChart(...)

### LLM Word-Count Consistency

**Scope:** LLM services only (OpenAI, Claude, Gemini, DeepSeek, Llama, Gemma, Qwen, Mistral), all 4 prompt variants. Sign languages are included and appear as **"Sign languages (gloss)"** in the charts — LLM output for signed languages is typically a Latin-script gloss or description rather than a lexical translation, so high CV for that row reflects inconsistency in *output strategy* (some models write one word, others write a sentence) rather than vocabulary uncertainty within the family.

The charts below show, from broadest to finest grain: overall CV per family → CV by service and by variant → the full family × variant heatmap.

In [12]:
import numpy as np

term = TARGET_TERMS[0]
full_df = all_dfs[term].copy()
full_df["language_family"] = full_df["language_code"].apply(get_language_family)

LLM_ONLY = {s: c for s, c in HEATMAP_SERVICES.items() if s in LLM_SERVICES}

wc_rows = []
for service, col in LLM_ONLY.items():
    if col not in full_df.columns:
        continue
    mask = full_df[col].notna()
    sub = full_df.loc[mask, ["language_code", "language_name", "language_family", "prompt_variant"]].copy()
    sub["word_count"] = full_df.loc[mask, col].astype(str).str.split().str.len().values
    sub["service"] = service
    wc_rows.append(sub)

wc_long = pd.concat(wc_rows, ignore_index=True)

# Sign languages are included but labelled "(gloss)" to signal that LLM output
# for signed languages is a Latin-script gloss or description, not a lexical
# translation. High CV for this row reflects output-strategy inconsistency, not
# vocabulary uncertainty — see section header for discussion.
wc_long["family_grouped"] = wc_long["language_family"].replace(
    {"Sign languages": "Sign languages (gloss)"}
)

In [13]:
var_rows = []
for (family, service, variant), grp in wc_long.groupby(["family_grouped", "service", "prompt_variant"]):
    mean_wc = grp["word_count"].mean()
    std_wc  = grp["word_count"].std()
    cv      = std_wc / mean_wc if mean_wc > 0 else np.nan
    var_rows.append({
        "family":  family,
        "service": service,
        "variant": variant,
        "n":       len(grp),
        "mean_wc": round(mean_wc, 2),
        "std_wc":  round(std_wc,  2),
        "cv":      round(cv,      3),
    })
var_df = pd.DataFrame(var_rows)

# ── Chart 1: overall CV per family (collapsed across service & variant) ──────
fam_cv = (
    wc_long.groupby("family_grouped")["word_count"]
    .agg(n="count", mean="mean", std="std")
    .assign(cv=lambda d: d["std"] / d["mean"])
    .sort_values("cv", ascending=False)
    .reset_index()
    .round(3)
)

fam_bar = alt.Chart(fam_cv).mark_bar().encode(
    y=alt.Y("family_grouped:N", sort=alt.EncodingSortField("cv", order="descending"), title=None),
    x=alt.X("cv:Q", title="CV (std / mean word count)"),
    color=alt.Color("cv:Q", scale=alt.Scale(scheme="reds"), legend=None),
    tooltip=["family_grouped:N",
             alt.Tooltip("cv:Q", format=".3f", title="CV"),
             alt.Tooltip("mean:Q", format=".2f", title="mean words"),
             alt.Tooltip("n:Q", title="translation rows")],
).properties(
    width=400,
    height=max(240, len(fam_cv) * 14),
    title="Word-count variability by family — collapsed across services & variants",
)

# ── Chart 2: mean CV by service and by variant ────────────────────────────────
svc_cv = var_df.groupby("service")["cv"].mean().reset_index().sort_values("cv", ascending=False)
var_cv = var_df.groupby("variant")["cv"].mean().reset_index().sort_values("cv", ascending=False)

svc_bar = alt.Chart(svc_cv).mark_bar().encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("cv", order="descending"), title=None),
    x=alt.X("cv:Q", title="mean CV", scale=alt.Scale(domain=[0, svc_cv["cv"].max() * 1.2])),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N", alt.Tooltip("cv:Q", format=".3f")],
).properties(width=240, height=160, title="Mean CV by service")
svc_text = svc_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text=alt.Text("cv:Q", format=".3f"), color=alt.value("black"))

var_bar = alt.Chart(var_cv).mark_bar().encode(
    y=alt.Y("variant:N", sort=alt.EncodingSortField("cv", order="descending"), title=None),
    x=alt.X("cv:Q", title="mean CV", scale=alt.Scale(domain=[0, var_cv["cv"].max() * 1.2])),
    color=alt.Color("variant:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["variant:N", alt.Tooltip("cv:Q", format=".3f")],
).properties(width=240, height=160, title="Mean CV by prompt variant")
var_text = var_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text=alt.Text("cv:Q", format=".3f"), color=alt.value("black"))

fam_bar.display()
((svc_bar + svc_text) | (var_bar + var_text)).display()

alt.Chart(...)

alt.HConcatChart(...)

### Median CV Heatmap — Family × Variant

Rows sorted so high-variability families (where models are least grounded) land at the top. Structured variants (`fluent_speaker`, `judge`) consistently reduce CV — the reddest rows are where prompt scaffolding has the most leverage. All families appear individually; see §1.3 for the full family-size distribution.

In [14]:
# Median CV heatmap: language family × prompt variant.
# Rows sorted by CV so the high-variability families land at the top —
# this is the paper figure showing "prompt scaffolding helps most where
# models are least grounded."

fam_variant_cv = (
    var_df.groupby(["family", "variant"])["cv"]
    .median()
    .reset_index()
)

# Row order: by mean CV across variants, highest first
fam_order_cv = (
    fam_variant_cv.groupby("family")["cv"].mean()
    .sort_values(ascending=False).index.tolist()
)
variant_order_heat = ["minimal", "fluent_speaker", "github_searcher", "judge"]

heat = alt.Chart(fam_variant_cv).mark_rect().encode(
    x=alt.X("variant:N", sort=variant_order_heat, title="Prompt variant",
            axis=alt.Axis(labelAngle=-20)),
    y=alt.Y("family:N", sort=fam_order_cv, title="Language family"),
    color=alt.Color("cv:Q", scale=alt.Scale(scheme="reds"),
                    title="Median CV"),
    tooltip=["family:N", "variant:N",
             alt.Tooltip("cv:Q", format=".3f")],
).properties(width=320, height=300)

# Value labels. Use black/white contrast thresholded on CV.
text = alt.Chart(fam_variant_cv).mark_text(fontSize=10).encode(
    x=alt.X("variant:N", sort=variant_order_heat),
    y=alt.Y("family:N", sort=fam_order_cv),
    text=alt.Text("cv:Q", format=".2f"),
    color=alt.condition(
        alt.datum.cv > 0.4, alt.value("white"), alt.value("black")
    ),
)

(heat + text).properties(
    title="Translation word-count CV by family × prompt variant (lower = more consensus)"
)

alt.LayerChart(...)

## 2.3 Data Quality and Curation

Three interlocking quality signals — missing rationales, script anomalies, and script disagreements — are surfaced here and consolidated into a per-language quality-flags file used by the review explorer.

### Missing Rationales

LLM services are expected to return both a translated term *and* a rationale explaining their choice. Two failure modes arise in practice:

- **Missing rationale** — the service returned a translation but left the rationale field null or empty.
- **Placeholder rationale** — the model returned a literal string such as `"No rationale provided"` instead of real reasoning.

Both are treated as equivalent to a missing rationale. `load_variant_df`(`scripts/exploration/explore_confidence_within_variant.py`) calls `enforce_translation_rationale_pairing` from `scripts/utils.py` before returning, which nulls out:
- any translation whose rationale is absent or a placeholder, and
- any rationale (or placeholder) whose translation is absent.

This means every notebook and script that calls `load_variant_df` — including this one — already receives clean, paired data. `explore_disagreements.py` additionally drops any LLM service translation that has no real rationale in the loaded variant file, so unpaired rows cannot influence the disagreement classifier.

The table below counts mismatches *before* pairing enforcement, showing the raw scope of the problem per service × prompt variant.

In [15]:
LLM_RAT_COLS = {
    "Claude":   "claude_translation_rationale",
    "OpenAI":   "openai_translation_rationale",
    "Gemini":   "gemini_translation_rationale",
    "DeepSeek": "deepseek_translation_rationale",
    "Llama":    "llama_translation_rationale",
    "Gemma":    "gemma_translation_rationale",
    "Qwen":     "qwen_translation_rationale",
    "Mistral":  "mistral_translation_rationale",
}

mismatch_rows = []
df = all_dfs[TARGET_TERMS[0]]

for variant in VARIANTS:
    vdf = df[df["prompt_variant"] == variant]
    for service, trans_col in LLM_SERVICES.items():
        rat_col = LLM_RAT_COLS.get(service)
        if trans_col not in vdf.columns or rat_col not in vdf.columns:
            continue
        has_trans = vdf[trans_col].notna() & ~vdf[trans_col].astype(str).str.strip().isin(["", "nan"])
        has_rat   = vdf[rat_col].notna()   & ~vdf[rat_col].astype(str).str.strip().isin(["", "nan"])
        t_no_r = int((has_trans & ~has_rat).sum())
        r_no_t = int((~has_trans & has_rat).sum())
        mismatch_rows.append({
            "service":             service,
            "variant":             variant,
            "n_translation":       int(has_trans.sum()),
            "n_rationale":         int(has_rat.sum()),
            "trans_no_rationale":  t_no_r,
            "rationale_no_trans":  r_no_t,
        })

mismatch_df = pd.DataFrame(mismatch_rows)

def _highlight_nonzero(val):
    return "background-color: #ffe0e0" if isinstance(val, int) and val > 0 else ""

display(
    mismatch_df.style
    .map(_highlight_nonzero, subset=["trans_no_rationale", "rationale_no_trans"])
    .format({
        "n_translation":      "{:,}",
        "n_rationale":        "{:,}",
        "trans_no_rationale": "{:,}",
        "rationale_no_trans": "{:,}",
    })
    .set_caption("LLM service translation ↔ rationale mismatches (red = non-zero; these rows are excluded from explore_disagreements.py)")
)

,service,variant,n_translation,n_rationale,trans_no_rationale,rationale_no_trans
0,OpenAI,minimal,796,796,0,0
1,Claude,minimal,880,880,0,0
2,Gemini,minimal,845,847,0,2
3,DeepSeek,minimal,866,869,0,3
4,Llama,minimal,786,786,0,0
5,Gemma,minimal,864,864,0,0
6,Qwen,minimal,874,876,0,2
7,Mistral,minimal,828,828,1,1
8,OpenAI,expert_persona,844,844,0,0
9,Claude,expert_persona,880,880,0,0


### Script and Directionality

RTL languages and CJK-script languages are structurally harder for translation pipelines. Does coverage drop for these groups?

In [16]:
# Build script-group sets from the comprehensive language metadata
_lang_meta = read_csv_file(os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv"))
_lang_meta = _lang_meta.dropna(subset=["language_code"])

# RTL: CLDR-derived directionality column (78 codes); already has FORCE_LTR overrides applied
# — more complete than any hardcoded list and correctly excludes diq/ha/ku/uz/uz_AF
RTL_CODES = set(_lang_meta.loc[_lang_meta["directionality"] == "rtl", "language_code"])

# CJK / SE Asian: derive from primary_script
_CJK_SCRIPTS = {
    "Bopomofo", "Simplified", "Traditional", "Japanese", "Korean", "Katakana",
    "Tibetan", "Myanmar", "Khmer", "Lao", "Thai", "Han",
}
_cjk_from_csv = set(_lang_meta.loc[_lang_meta["primary_script"].isin(_CJK_SCRIPTS), "language_code"].dropna())
# Chinese variant codes present in pipeline data but missing primary_script in the metadata CSV
_CJK_EXTRA = {"zh-tw", "zh-classical", "zh-min-nan", "zh-yue", "cdo"}
CJK_CODES = _cjk_from_csv | _CJK_EXTRA

print(f"RTL codes: {len(RTL_CODES)}  |  CJK/SE Asian codes: {len(CJK_CODES)}")

def script_group(code):
    if code in RTL_CODES:
        return "RTL"
    if code in CJK_CODES:
        return "CJK / SE Asian"
    return "LTR (Latin/other)"

term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
svc_cols = {s: c for s, c in HEATMAP_SERVICES.items() if c in baseline.columns}
baseline["n_services"] = baseline[[c for c in svc_cols.values()]].notna().sum(axis=1)
baseline["script"] = baseline["language_code"].apply(script_group)

script_rows = []
for service, col in svc_cols.items():
    for script, grp in baseline.groupby("script"):
        n = int(grp[col].notna().sum())
        total = len(grp)
        script_rows.append({"service": service, "script": script, "n": n, "total": total})
script_df = pd.DataFrame(script_rows)

script_order = ["LTR (Latin/other)", "RTL", "CJK / SE Asian"]
svc_order = list(HEATMAP_SERVICES.keys())
max_n_script = script_df["n"].max()

heatmap = alt.Chart(script_df).mark_rect().encode(
    x=alt.X("service:N", sort=svc_order, title=None),
    y=alt.Y("script:N", sort=script_order, title=None),
    color=alt.Color("n:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, max_n_script]),
        title="languages translated"),
    tooltip=["service:N","script:N",
             alt.Tooltip("n:Q",title="translated"),
             alt.Tooltip("total:Q",title="total in group")],
)
hmap_text = heatmap.mark_text(fontSize=10).encode(
    text="n:Q",
    color=alt.condition(alt.datum.n > max_n_script * 0.6, alt.value("white"), alt.value("black")),
)

box = alt.Chart(baseline).mark_boxplot(extent="min-max").encode(
    x=alt.X("script:N", sort=script_order, title=None),
    y=alt.Y("n_services:Q", title="services that translated it", scale=alt.Scale(domain=[0,9])),
    color=alt.Color("script:N", sort=script_order, legend=None),
    tooltip=["script:N"],
).properties(width=260, height=220, title="Coverage depth by script group")

(heatmap + hmap_text).properties(
    width=480, height=100,
    title="Languages translated per service by script group",
) & box

RTL codes: 78  |  CJK/SE Asian codes: 12


alt.VConcatChart(...)

#### Script Agreement Across LLM Services

How often do the eight LLM services agree on *which script* to use for a given language? Full agreement is the norm for well-resourced languages, but disagreement reveals where models are uncertain about the target orthography — or are silently romanising rather than generating the target script.

The second half of this section compares local Ollama models (Llama, Gemma, Qwen, Mistral) against API models (Claude, OpenAI, Gemini, DeepSeek) to see which category is more often the source of script disagreements.

In [17]:
from collections import Counter
from scripts.utils import detect_dominant_script

LLM_SCRIPT_COLS = {
    'Claude':   'claude_translated_term',
    'OpenAI':   'openai_translated_term',
    'Gemini':   'gemini_translated_term',
    'DeepSeek': 'deepseek_translated_term',
    'Llama':    'llama_translated_term',
    'Gemma':    'gemma_translated_term',
    'Qwen':     'qwen_translated_term',
    'Mistral':  'mistral_translated_term',
}

df = all_dfs[TARGET_TERMS[0]]

# ── Part 1: general script agreement across all 8 LLMs ───────────────────────
agreement_rows = []
outlier_rows = []

for variant in VARIANTS:
    vdf = df[df['prompt_variant'] == variant]
    for _, row in vdf.iterrows():
        lang = row['language_code']
        fam  = get_language_family(lang)
        scripts = {}
        for svc, col in LLM_SCRIPT_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                s = detect_dominant_script(str(val))
                if s != 'Unknown':
                    scripts[svc] = s
        if len(scripts) < 2:
            continue

        n_distinct = len(set(scripts.values()))
        agreement_rows.append({
            'language_code':     lang,
            'language_name':     row.get('language_name', lang),
            'language_family':   fam,
            'variant':           variant,
            'n_services':        len(scripts),
            'n_distinct_scripts': n_distinct,
        })

        if n_distinct > 1:
            majority_script = Counter(scripts.values()).most_common(1)[0][0]
            for svc, script in scripts.items():
                if script != majority_script:
                    outlier_rows.append({
                        'language_code':   lang,
                        'language_family': fam,
                        'variant':         variant,
                        'service':         svc,
                        'majority_script': majority_script,
                        'outlier_script':  script,
                    })

agree_df   = pd.DataFrame(agreement_rows)
outlier_df = pd.DataFrame(outlier_rows)

n_rows     = len(agree_df)
n_disagree = (agree_df['n_distinct_scripts'] > 1).sum()
print(f"Rows with ≥2 LLM translations: {n_rows:,}")
print(f"Any script disagreement: {n_disagree:,} ({n_disagree/n_rows*100:.1f}%)")
print(f"Full agreement (all same script): {(agree_df['n_distinct_scripts']==1).sum():,} "
      f"({(agree_df['n_distinct_scripts']==1).mean()*100:.1f}%)")

# Chart 1: distribution of n_distinct_scripts per row
label_map = {1: '1 — all agree', 2: '2 scripts', 3: '3 scripts', 4: '4 scripts',
             5: '5 scripts', 6: '6 scripts', 7: '7 scripts', 8: '8 scripts'}
dist = (
    agree_df['n_distinct_scripts'].value_counts().reset_index()
    .rename(columns={'n_distinct_scripts': 'n_distinct', 'count': 'count'})
)
dist['pct']   = (dist['count'] / n_rows * 100).round(1)
dist['label'] = dist['n_distinct'].map(label_map)
label_order   = [label_map[i] for i in sorted(label_map) if i in dist['n_distinct'].values]

dist_bar = alt.Chart(dist).mark_bar().encode(
    x=alt.X('label:N', sort=label_order, title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count:Q', title='rows (language × variant)'),
    color=alt.condition(
        alt.datum.n_distinct == 1, alt.value('#388e3c'), alt.value('#d32f2f')),
    tooltip=['label:N', 'count:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(width=300, height=220, title='Script agreement across 8 LLM services')
dist_text = dist_bar.mark_text(dy=-6, fontSize=11).encode(
    text=alt.Text('pct:Q', format='.1f'), color=alt.value('black'))

# Chart 2: which service is most often the outlier?
svc_outlier = (
    outlier_df.groupby('service').size().reset_index(name='n')
    .sort_values('n', ascending=False)
)
svc_outlier['pct'] = (svc_outlier['n'] / len(outlier_df) * 100).round(1)

outlier_bar = alt.Chart(svc_outlier).mark_bar().encode(
    y=alt.Y('service:N', sort=alt.EncodingSortField('n', order='descending'), title=None),
    x=alt.X('n:Q', title='times as script outlier'),
    color=alt.Color('service:N', legend=None, scale=alt.Scale(scheme='tableau10')),
    tooltip=['service:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='% of all outlier events')],
).properties(width=300, height=200, title='Which service is most often the script outlier?')
outlier_text = outlier_bar.mark_text(align='left', dx=4, fontSize=9).encode(
    text='n:Q', color=alt.value('black'))

# Chart 3: families with most script disagreement
fam_disagree = (
    agree_df[agree_df['n_distinct_scripts'] > 1]
    .groupby('language_family')['language_code']
    .nunique().reset_index(name='n_langs')
    .sort_values('n_langs', ascending=False).head(12)
)
fam_dis_bar = alt.Chart(fam_disagree).mark_bar(color='#1976d2').encode(
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField('n_langs', order='descending'), title=None),
    x=alt.X('n_langs:Q', title='unique languages with any script disagreement'),
    tooltip=['language_family:N', 'n_langs:Q'],
).properties(width=300, height=260, title='Families with most script disagreement')
fam_dis_text = fam_dis_bar.mark_text(align='left', dx=4, fontSize=9).encode(
    text='n_langs:Q', color=alt.value('black'))

(dist_bar + dist_text).display()
((outlier_bar + outlier_text) | (fam_dis_bar + fam_dis_text)).display()

# ── Part 2: Local vs API model script outlier analysis ───────────────────────
LOCAL_MODELS = ['Llama', 'Gemma', 'Qwen', 'Mistral']
API_MODELS   = ['Claude', 'OpenAI', 'Gemini', 'DeepSeek']

local_script_rows = []

for variant in VARIANTS:
    vdf = df[df['prompt_variant'] == variant]
    for _, row in vdf.iterrows():
        scripts = {}
        for svc, col in LLM_SCRIPT_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                s = detect_dominant_script(str(val))
                if s != 'Unknown':
                    scripts[svc] = s

        api_scripts = {s: scripts[s] for s in API_MODELS if s in scripts}
        if len(api_scripts) < 2:
            continue
        majority_script = Counter(api_scripts.values()).most_common(1)[0][0]
        majority_count  = list(api_scripts.values()).count(majority_script)
        if majority_count < 2:
            continue

        for local_svc in LOCAL_MODELS:
            if local_svc not in scripts:
                continue
            if scripts[local_svc] != majority_script:
                local_script_rows.append({
                    'variant':         variant,
                    'language_code':   row['language_code'],
                    'language_name':   row.get('language_name', row['language_code']),
                    'language_family': get_language_family(row['language_code']),
                    'local_service':   local_svc,
                    'local_script':    scripts[local_svc],
                    'majority_script': majority_script,
                    'api_agree_n':     majority_count,
                })

local_sd = pd.DataFrame(local_script_rows)

print(f"\nLocal model script outliers (API majority ≥2 agree on different script): "
      f"{len(local_sd)} rows, {local_sd['language_code'].nunique() if len(local_sd) else 0} unique languages")

if len(local_sd) > 0:
    # Chart 4: which local model is most often the script outlier?
    svc_counts = (
        local_sd.groupby('local_service').size().reset_index(name='n')
        .sort_values('n', ascending=False)
    )
    svc_counts['pct'] = (svc_counts['n'] / len(local_sd) * 100).round(1)

    svc_bar = alt.Chart(svc_counts).mark_bar().encode(
        y=alt.Y('local_service:N', sort=alt.EncodingSortField('n', order='descending'), title=None),
        x=alt.X('n:Q', title='times as script outlier vs API majority'),
        color=alt.Color('local_service:N', legend=None, scale=alt.Scale(scheme='tableau10')),
        tooltip=['local_service:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='% of local outlier events')],
    ).properties(width=300, height=160, title='Which local model deviates most from API script consensus?')
    svc_text = svc_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n:Q', color=alt.value('black'))

    # Chart 5: script-pair breakdown (API majority → local model's script)
    pair_counts = (
        local_sd.groupby(['majority_script', 'local_script'])
        .size().reset_index(name='n')
        .sort_values('n', ascending=False).head(12)
    )
    pair_counts['pair'] = pair_counts['majority_script'] + ' → ' + pair_counts['local_script']

    pair_bar = alt.Chart(pair_counts).mark_bar().encode(
        y=alt.Y('pair:N', sort='-x', title=None),
        x=alt.X('n:Q', title='occurrences (across all variants)'),
        color=alt.Color('local_script:N', title="local model's script",
                        scale=alt.Scale(scheme='tableau10')),
        tooltip=['pair:N', 'n:Q', 'majority_script:N', 'local_script:N'],
    ).properties(width=380, height=280,
                 title="Local model script mismatches: API majority script → local model's script")
    pair_text = pair_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n:Q', color=alt.value('black'))

    # Chart 6: top families in local model outlier cases
    fam_local = (
        local_sd.groupby('language_family')['language_code']
        .nunique().reset_index(name='n_langs')
        .sort_values('n_langs', ascending=False).head(10)
    )
    fam_loc_bar = alt.Chart(fam_local).mark_bar(color='#8b00d4').encode(
        y=alt.Y('language_family:N',
                sort=alt.EncodingSortField('n_langs', order='descending'), title=None),
        x=alt.X('n_langs:Q', title='unique languages'),
        tooltip=['language_family:N', 'n_langs:Q'],
    ).properties(width=280, height=240, title='Families most affected by local model script switching')
    fam_loc_text = fam_loc_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n_langs:Q', color=alt.value('black'))

    (svc_bar + svc_text).display()
    (pair_bar + pair_text).display()
    (fam_loc_bar + fam_loc_text).display()
else:
    print("No local model script outlier cases found.")


Rows with ≥2 LLM translations: 3,520
Any script disagreement: 1,076 (30.6%)
Full agreement (all same script): 2,444 (69.4%)


alt.LayerChart(...)

alt.HConcatChart(...)


Local model script outliers (API majority ≥2 agree on different script): 1932 rows, 363 unique languages


alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

#### Script Disagreement by Language Family

The overall script-disagreement rate (378/880 = 43.0%) is not uniform across language families. Two patterns are worth flagging:

**High rate, low expected script count** — families where most languages have a single established orthography but models still produce script variation. This is model-injected noise rather than genuine linguistic ambiguity. Tai languages (100%), Sino-Tibetan (93.6%), Eskimo-Aleut (80%), and Nilo-Saharan (51.7%) fall here. Gemma is the primary driver: it generates transliterations in a secondary script (often Latin) alongside or instead of the primary script.

**High expected script count, no disagreement** — 40 languages with 2+ registered scripts where all 8 LLMs nonetheless agreed on a single script. These are typically languages with one dominant script in digital text (the minority script is historical or liturgical), so model convergence is correct rather than coincidental.


In [18]:
# Script disagreement by language family
# agree_df and outlier_df are built in the cell above (Script Agreement Across LLM Services)
# _lang_meta is built in the Script and Directionality setup cell

_scripts_per_lang = (
    _lang_meta[["language_code", "n_scripts"]]
    .dropna(subset=["language_code"])
    .copy()
)

# Summarize at language level: disagreement = any variant had n_distinct_scripts > 1
_lang_disagr = (
    agree_df.groupby(["language_code", "language_family"])
    .agg(has_script_disagr=("n_distinct_scripts", lambda x: (x > 1).any()))
    .reset_index()
)
_lang_disagr = _lang_disagr.merge(_scripts_per_lang, on="language_code", how="left")

fam_stats = (
    _lang_disagr.groupby("language_family")
    .agg(
        n_langs=("language_code", "nunique"),
        n_disagr=("has_script_disagr", "sum"),
        avg_n_scripts=("n_scripts", "mean"),
    )
    .reset_index()
)
fam_stats["rate"] = fam_stats["n_disagr"] / fam_stats["n_langs"]
fam_stats = fam_stats.sort_values("rate", ascending=False)

chart = alt.Chart(fam_stats[fam_stats["n_langs"] >= 5]).mark_bar().encode(
    y=alt.Y("language_family:N", sort="-x", title=None),
    x=alt.X("rate:Q", axis=alt.Axis(format="%"), title="script disagreement rate"),
    color=alt.Color("avg_n_scripts:Q",
                    scale=alt.Scale(scheme="blues", domainMin=1),
                    title="avg registered scripts"),
    tooltip=[
        "language_family:N",
        alt.Tooltip("rate:Q", format=".1%"),
        alt.Tooltip("n_langs:Q", title="languages"),
        alt.Tooltip("n_disagr:Q", title="disagreements"),
        alt.Tooltip("avg_n_scripts:Q", format=".2f", title="avg scripts"),
    ],
).properties(width=420, height=280,
             title="Script disagreement rate by family (n ≥ 5 languages)")
display(chart)

# Drill into top 3 high-rate / low avg_n_scripts families (model-injected noise)
_high_rate = fam_stats[
    (fam_stats["n_langs"] >= 5) &
    (fam_stats["rate"] > 0.5) &
    (fam_stats["avg_n_scripts"] < 1.3)
].head(3)

print("High script-disagreement rate, low expected script diversity (model-injected noise):")
for _, frow in _high_rate.iterrows():
    fam = frow["language_family"]
    print(f"\n  {fam}  rate={frow['rate']:.0%}, {int(frow['n_langs'])} langs, "
          f"avg {frow['avg_n_scripts']:.2f} scripts/lang")
    # Outlier services for this family
    fam_out = (
        outlier_df[outlier_df["language_family"] == fam]
        .groupby("service")["language_code"]
        .nunique()
        .sort_values(ascending=False)
    )
    print(f"  Outlier services: {dict(fam_out)}")
    # Example languages
    ex_langs = (
        _lang_disagr[
            (_lang_disagr["language_family"] == fam) &
            (_lang_disagr["has_script_disagr"])
        ][["language_code"]]
        .drop_duplicates()
        .merge(
            all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]].drop_duplicates("language_code"),
            on="language_code", how="left"
        )
        .head(5)
    )
    for _, r in ex_langs.iterrows():
        print(f"    {r['language_code']} ({r['language_name']})")

# Languages with n_scripts > 1 but no disagreement (model correctly converges)
_multi_nodisr = _lang_disagr[
    (_lang_disagr["n_scripts"] > 1) & (~_lang_disagr["has_script_disagr"])
]
print(f"\nLanguages with ≥2 registered scripts but no cross-service disagreement: {len(_multi_nodisr)}")
print("(Models converge on dominant digital-text script despite multiple registered orthographies)")


alt.Chart(...)

High script-disagreement rate, low expected script diversity (model-injected noise):

  Tai-Kadai languages  rate=100%, 10 langs, avg 1.10 scripts/lang
  Outlier services: {'Gemma': np.int64(8), 'Llama': np.int64(8), 'Mistral': np.int64(5), 'Qwen': np.int64(4), 'Claude': np.int64(3), 'Gemini': np.int64(3), 'OpenAI': np.int64(3), 'DeepSeek': np.int64(2)}
    blt (Tai Dam)
    khb (Lü)
    lo (Lao)
    nod (Northern Thai)
    shn (Shan)

  Sino-Tibetan languages  rate=92%, 51 langs, avg 1.22 scripts/lang
  Outlier services: {'Gemma': np.int64(38), 'Llama': np.int64(36), 'Qwen': np.int64(27), 'Mistral': np.int64(26), 'Gemini': np.int64(16), 'DeepSeek': np.int64(14), 'Claude': np.int64(13), 'OpenAI': np.int64(11)}
    bap (Bantawa)
    bft (Balti)
    bo (Tibetan)
    brx (Boro)
    ccp (Chakma)

  Eskimo-Aleut languages  rate=80%, 5 langs, avg 1.00 scripts/lang
  Outlier services: {'Mistral': np.int64(4), 'Gemini': np.int64(3), 'Gemma': np.int64(3), 'Qwen': np.int64(3), 'Claude': np.int64

#### Mixed-Script Translations

A subtler script anomaly: individual translations that contain characters from two or more distinct scripts within a single string. For example, a Cyrillic-target language that receives "Цифровые Humanities" mixes Cyrillic and Latin in one output. This can indicate a model giving up mid-word, defaulting to an English fragment, or failing to transliterate a technical term.

Detection rule: a translation is "mixed" when a secondary script accounts for ≥ 10 % of all script-significant characters. CJK and Japanese syllabaries are treated as one family (Hiragana/Katakana + Kanji co-occurrence is normal). Services with no prompt variation (Wikipedia, Google Translate, EasyNMT, Lingvanex) use the minimal-variant row only.

In [19]:
import unicodedata as _ud
from scripts.utils import _char_script

MIXED_THRESHOLD = 0.10  # secondary script must be ≥10% of script chars to count

def _script_profile(text):
    """Return {script: char_count}, ignoring punctuation/digits/spaces."""
    if not isinstance(text, str) or not text.strip():
        return {}
    counts = Counter()
    for ch in text:
        cp = ord(ch)
        cat = _ud.category(ch)
        if cat[0] in ('Z', 'P', 'S', 'C') or cat == 'Nd':
            continue
        s = _char_script(cp)
        if s == 'Other':
            continue
        # CJK kanji + Japanese kana co-occur normally in Japanese — merge them
        s = 'CJK/Japanese' if s in ('CJK', 'Japanese') else s
        counts[s] += 1
    return dict(counts)

def _is_mixed(profile, threshold=MIXED_THRESHOLD):
    if len(profile) < 2:
        return False
    total = sum(profile.values())
    dominant = max(profile.values())
    return total > 0 and (total - dominant) / total >= threshold

# ── Build mixed-script rows across all services and variants ─────────────────
ALL_SERVICES = {**BASELINE_SERVICES, **LLM_SERVICES}
term = TARGET_TERMS[0]
df_all = all_dfs[term].copy()
df_all["language_family"] = df_all["language_code"].apply(get_language_family)

# For baseline services use minimal variant only (they don't vary by prompt)
baseline_rows_df = df_all[df_all["prompt_variant"] == "minimal"].copy()

mixed_rows = []
for service, col in ALL_SERVICES.items():
    if col not in df_all.columns:
        continue
    src = baseline_rows_df if service in BASELINE_SERVICES else df_all
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        profile = _script_profile(val)
        if not profile:
            continue
        total_chars = sum(profile.values())
        scripts_present = sorted(profile, key=lambda s: -profile[s])
        dominant = scripts_present[0]
        minority_frac = (total_chars - profile[dominant]) / total_chars
        mixed_rows.append({
            "service":         service,
            "language_code":   row["language_code"],
            "language_name":   row.get("language_name", row["language_code"]),
            "language_family": row["language_family"],
            "variant":         row["prompt_variant"],
            "translation":     val,
            "dominant_script": dominant,
            "n_scripts":       len(profile),
            "minority_frac":   round(minority_frac, 3),
            "is_mixed":        _is_mixed(profile),
            "scripts_str":     " + ".join(scripts_present),
        })

mixed_df = pd.DataFrame(mixed_rows)

print(f"Total translations checked: {len(mixed_df):,}")
print(f"Mixed-script ({MIXED_THRESHOLD*100:.0f}% threshold): {mixed_df['is_mixed'].sum():,} "
      f"({mixed_df['is_mixed'].mean()*100:.1f}%)")
print()
print(mixed_df[mixed_df['is_mixed']].groupby('service')['is_mixed'].sum().sort_values(ascending=False).to_string())

# ── Chart 1: % mixed-script by service ───────────────────────────────────────
svc_mix = (
    mixed_df.groupby("service")
    .agg(n_total=("is_mixed", "count"), n_mixed=("is_mixed", "sum"))
    .assign(pct_mixed=lambda d: (d["n_mixed"] / d["n_total"] * 100).round(1))
    .reset_index()
    .sort_values("pct_mixed", ascending=False)
)
svc_bar = alt.Chart(svc_mix).mark_bar().encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("pct_mixed", order="descending"), title=None),
    x=alt.X("pct_mixed:Q", title="% translations with mixed script"),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N",
             alt.Tooltip("pct_mixed:Q", format=".1f", title="% mixed"),
             alt.Tooltip("n_mixed:Q", title="mixed count"),
             alt.Tooltip("n_total:Q", title="total")],
).properties(width=340, height=220, title="Mixed-script translations by service")
svc_text = svc_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text=alt.Text("pct_mixed:Q", format=".1f"), color=alt.value("black"))

# ── Chart 2: dominant script pairs (what gets mixed with what) ───────────────
only_mixed = mixed_df[mixed_df["is_mixed"]].copy()
pair_counts = (
    only_mixed.groupby(["scripts_str", "service"])
    .size().reset_index(name="n")
    .sort_values("n", ascending=False)
)
top_pairs = pair_counts.groupby("scripts_str")["n"].sum().nlargest(10).index
pair_top = pair_counts[pair_counts["scripts_str"].isin(top_pairs)]

pair_bar = alt.Chart(pair_top).mark_bar().encode(
    y=alt.Y("scripts_str:N", sort=alt.EncodingSortField("n", order="descending"), title=None),
    x=alt.X("n:Q", title="occurrences"),
    color=alt.Color("service:N", scale=alt.Scale(scheme="tableau10"), title="Service"),
    tooltip=["scripts_str:N", "service:N", "n:Q"],
).properties(width=380, height=240, title="Most common script mixtures (top 10 pairs)")

# ── Chart 3: affected language families ──────────────────────────────────────
fam_mix = (
    only_mixed.groupby("language_family")["language_code"]
    .nunique().reset_index(name="n_langs")
    .sort_values("n_langs", ascending=False).head(12)
)
fam_bar = alt.Chart(fam_mix).mark_bar(color="#6a1b9a").encode(
    y=alt.Y("language_family:N", sort=alt.EncodingSortField("n_langs", order="descending"), title=None),
    x=alt.X("n_langs:Q", title="unique languages with mixed-script output"),
    tooltip=["language_family:N", "n_langs:Q"],
).properties(width=300, height=240, title="Language families with most mixed-script languages")
fam_text = fam_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text="n_langs:Q", color=alt.value("black"))

(svc_bar + svc_text).display()
(pair_bar | (fam_bar + fam_text)).display()

# ── Sample mixed-script translations ─────────────────────────────────────────
print("\nSample mixed-script translations (≥25% minority script):")
samples = (
    only_mixed[only_mixed["minority_frac"] >= 0.25]
    [["service","language_name","dominant_script","scripts_str","minority_frac","translation"]]
    .sort_values(["service","minority_frac"], ascending=[True, False])
    .groupby("service").head(3)
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", 60)
display(samples)

Total translations checked: 27,275
Mixed-script (10% threshold): 508 (1.9%)

service
Gemma               242
Qwen                108
Mistral              58
Llama                45
Gemini               33
Claude               14
DeepSeek              4
OpenAI                3
Google Translate      1


alt.LayerChart(...)

alt.HConcatChart(...)


Sample mixed-script translations (≥25% minority script):


,service,language_name,dominant_script,scripts_str,minority_frac,translation
0,Claude,mey,Latin,Latin + Arabic,0.449,al-ʿulūm al-insāniyya ar-raqmiyya / العلوم الإنسانية الر...
1,Claude,Ancient North Arabian,Latin,Latin + Arabic,0.286,ḥkmت ṣnʿت
2,Claude,Ancient North Arabian,Latin,Latin + Arabic,0.286,ḥkmت ṣnʿت
3,DeepSeek,Sogdian,Latin,Latin + Greek,0.250,δβ'nyk 'nš'n-δ'nšn'
4,Gemini,Classical Newari,Devanagari,Devanagari + Latin,0.500,अङ्कीय मानवशास्त्र (Aṅkīya Mānavśāstra)
5,Gemini,Manipuri,Bengali,Bengali + Latin,0.500,ডিজিটেল হিউমেনিতিজ (Digital Humanities)
6,Gemini,Naxi,Latin,Latin + Tibetan,0.486,གློག་རྡུལ་མི་ཆོས་རིག་པ། (Gle-rdeu mi-chos rig-pa)
7,Gemma,Pāli,Thai,Thai + Devanagari + Malayalam,0.562,ดิจิทัล മാനवीय कला
8,Gemma,Kachin,Latin,Latin + Thai + Myanmar + Khmer + Sinhala,0.515,ඩිជីថល လူမှုศาสตร์ (Di-ji-thal lu-mu-saat)
9,Gemma,Burmese,Latin,Latin + Khmer + Myanmar,0.511,ឌစ်ជីថលមនុស្សជាតិវិជ្ជា (Di-ji-tha-l Ma-nu-za-hai Wi-zha)


#### Curating Mixed-Script Translations

 implements five classification outcomes:

- **Strip** (Pattern A): removes `(romanization)` parentheticals and slash-separated suffixes, then verifies the remainder is single-script. The curated term is used going forward.
- **Strip** (Pattern C): removes a source-term prefix before a colon separator (e.g. `"Digital Humanities : कार्यान्वित मानवशास्त्र"` → `कार्यान्वित मानवशास्त्र`). Only applied when the prefix contains no characters from the dominant (native) script, so native text with colons is never truncated.
- **Strip** (Pattern D): removes a source-term prefix separated only by whitespace (e.g. `"Digital Humanities के दिशा कौशल"` → `के दिशा कौशल`). Strips the leading run of tokens that share the script of the first token; only accepted if the remainder is single-script.
- **Strip** (Pattern E): removes a source-term prefix before an equals-sign separator (e.g. `"Digital Humanities = Panagbalikas iti Digital"` → `Panagbalikas iti Digital`). Unlike Patterns C and D, this check runs *before* the mixed-script gate — both sides may share the same script (Latin-script translations), so the string would otherwise pass through unchanged. Only applied when the prefix is entirely Latin.
- **Strip** (Pattern F): removes slash delimiters wrapping the entire term (e.g. `"/Dkawng Thaukhnawng/"` → `"Dkawng Thaukhnawng"`). Distinct from Pattern A's inline slash separator, which strips a suffix; Pattern F handles terms fully enclosed in leading and trailing slashes.
- **Null** (Pattern B): interleaved character noise that stripping cannot fix — treated as a failed translation.

The charts below show how many translations per service are affected, and a sample of before → after transformations.

In [20]:
from scripts.exploration.translation_classifier import curate_translation, curate_df

term = TARGET_TERMS[0]
raw_df = all_dfs[term].copy()

# Apply curation to all *_translated_term columns
curated_df, summary = curate_df(raw_df)

print("Curation summary per service:")
print(summary.to_string(index=False))

# ── Chart: stripped / nulled / placeholder counts by service ──────────────────
summary_long = summary.melt(
    id_vars="service", value_vars=["stripped", "nulled", "placeholder"],
    var_name="action", value_name="count"
)
summary_long = summary_long[summary_long["count"] > 0]

if summary_long.empty:
    print("\nNo mixed-script or placeholder translations found at current threshold.")
else:
    action_bar = alt.Chart(summary_long).mark_bar().encode(
        y=alt.Y("service:N", sort=alt.EncodingSortField("count", order="descending"), title=None),
        x=alt.X("count:Q", title="translations affected"),
        color=alt.Color("action:N",
            scale=alt.Scale(
                domain=["stripped", "nulled", "placeholder"],
                range=["#1976d2", "#d32f2f", "#e67e00"],
            ),
            title="Action"),
        row=alt.Row("action:N", title=None),
        tooltip=["service:N", "action:N", "count:Q"],
    ).properties(width=380, height=120)
    action_bar.display()

# ── Sample before → after table ───────────────────────────────────────────────
ALL_TERM_COLS = {**BASELINE_SERVICES, **LLM_SERVICES}
sample_rows = []
for service, col in ALL_TERM_COLS.items():
    if col not in raw_df.columns:
        continue
    for (_, raw_row), (_, clean_row) in zip(raw_df.iterrows(), curated_df.iterrows()):
        before = raw_row[col]
        after  = clean_row[col]
        import pandas as _pd
        if not isinstance(before, str):
            continue
        # Only show rows where something changed
        if before == after or (_pd.isna(after) and not isinstance(before, str)):
            continue
        if isinstance(before, str) and (after is None or _pd.isna(after) or before != after):
            _, action = curate_translation(before)
            if action == "unchanged":
                continue
            after_display = str(after) if after is not None and not _pd.isna(after) else f"— {action} —"
            sample_rows.append({
                "service":  service,
                "language": raw_row.get("language_name", raw_row["language_code"]),
                "action":   action,
                "before":   before,
                "after":    after_display,
            })

sample_df = (
    pd.DataFrame(sample_rows)
    .sort_values(["action", "service"])
    .groupby(["action", "service"]).head(2)
    .reset_index(drop=True)
)

if not sample_df.empty:
    print("\nSample transformations:")
    pd.set_option("display.max_colwidth", 70)
    display(sample_df[["service", "language", "action", "before", "after"]])

# ── Coverage impact: how many languages gain/lose coverage after curation ─────
svc_cols = [c for c in ALL_TERM_COLS.values() if c in raw_df.columns]
before_cov = raw_df.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any())
after_cov  = curated_df.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any())

impact_rows = []
for col, service in {v: k for k, v in ALL_TERM_COLS.items() if v in raw_df.columns}.items():
    lost = int((before_cov[col] & ~after_cov[col]).sum())
    impact_rows.append({"service": service, "languages_losing_coverage": lost})

impact_df = pd.DataFrame(impact_rows).sort_values("languages_losing_coverage", ascending=False)
print("\nLanguages losing coverage after nulling/placeholder curation:")
print(impact_df[impact_df["languages_losing_coverage"] > 0].to_string(index=False))


Curation summary per service:
  service  unchanged  stripped  nulled  placeholder
     enmt       3520         0       0            0
wikipedia       3520         0       0            0
       gt       3520         0       0            0
lingvanex       3520         0       0            0
   claude       3507         2       0           11
    llama       3477        23      20            0
    gemma       3303       180      37            0
     qwen       3440        25      55            0
  mistral       3459        47       8            6
   openai       3506         0       0           14
 deepseek       3518         0       0            2
   gemini       3490        20       3            7


alt.Chart(...)


Sample transformations:


,service,language,action,before,after
0,Gemini,Tachelhit,nulled,تودرت ن ⵓⵎⴰⵏ ⴷ ⵓⵎⴰⵏ ⴰⴷⵉⵊⵉⵜⴰⵍ,— nulled —
1,Gemini,Zaghawa,nulled,تِكْنُوجِيَا ʔɪ́ŋɡɪ́lɪ́ ʔɪ́ŋɡɪ́lɪ́,— nulled —
2,Gemma,Maba,nulled,ඩිජිටල් මානවwissenschaft,— nulled —
3,Gemma,Mongolian,nulled,Дижитал humanistууд,— nulled —
4,Llama,Urdu,nulled,ڈیجیٹل مानवیات,— nulled —
5,Llama,Saraiki,nulled,ڈیجیٹل مानवیات,— nulled —
6,Mistral,Tae',nulled,디지털 umanis,— nulled —
7,Mistral,Abkhaz,nulled,Информационные гumanitарные науки,— nulled —
8,Qwen,Russian,nulled,Дigitalnye гуманитарные науки,— nulled —
9,Qwen,Eastern Magar,nulled,द数-digitल अवस्थानीय विज्ञान,— nulled —



Languages losing coverage after nulling/placeholder curation:
Empty DataFrame
Columns: [service, languages_losing_coverage]
Index: []


### Quality Flags

Consolidate the data-quality signals surfaced in §2.3 (missing rationales, mixed-script, and script disagreement) into a single set of flags per language. 

Eleven binary flags per language:

- any LLM has a translation without a rationale (or vice versa) in any variant
- any service produced a translation with genuine script mixing (Pattern B from §1.8.3)
- any service produced a translation that was stripped (Patterns A/C/D/E/F from §1.8.3): romanization parenthetical, colon-separated source prefix, space-separated source prefix, equals-sign source prefix, or slash-wrapped term
- services used different scripts for this language (from §1.8.1)
- any service returned the untranslated English source term (e.g. `"Digital Humanities"` or `"DH"` as a standalone token) in a non-English translation
- any service returned a refusal or placeholder string instead of a real translation (e.g. `"untranslatable"`, `"no direct translation"`, `"fictional_translation"`, `"Note: ..."`)
- any service produced a translation where a single token repeats ≥4 times and represents ≥30% of all tokens (e.g. EasyNMT Vietnamese `"bình bình bình..."` × 17, Gemini Ngambay `"kàlā kàlā..."` × 5)
- any service produced a translation longer than 100 characters, catching both hallucination loops and LLM disclaimer text that slipped past placeholder detection
- any service produced a translation containing literal `\uXXXX` escape sequences instead of rendered Unicode characters
- any service produced a translation with fewer than 4 Unicode codepoints (e.g. a single character or syllable); these are too short for reliable GitHub search and tend to produce false positives regardless of script
- any service produced a translation with any secondary-script characters at all, regardless of whether they cross the exclusion threshold; this is a superset of  and  and captures sub-threshold mixing (e.g. Chechen palochka substitutes, Ossetian æ) as data rather than errors

In [21]:
from collections import defaultdict
from scripts.exploration.translation_classifier import (
    curate_translation, has_source_leakage,
    is_repetition_loop, has_extreme_term_length, has_short_translation, has_unicode_escape,
    script_mix_detail,
)

term = TARGET_TERMS[0]
df_raw = all_dfs[term].copy()

# ── 1. Missing rationale ─────────────────────────────────────────────────────
# all_dfs uses load_variant_df, which applies enforce_translation_rationale_pairing
# before returning — so mismatches are already erased there. Read the raw
# prompt_services CSVs directly to catch the original pairings.
_PLACEHOLDER_RATS = {
    "no rationale provided", "no rationale", "n/a", "none",
    "not applicable", "no explanation provided", "no reason provided",
}

def _has_rat(val):
    if not isinstance(val, str) or not val.strip():
        return False
    return val.strip().rstrip(".").lower() not in _PLACEHOLDER_RATS

_SVC_FILE_KEY = {
    "Claude": "claude", "OpenAI": "openai", "Gemini": "gemini", "DeepSeek": "deepseek",
    "Llama": "llama", "Gemma": "gemma", "Qwen": "qwen", "Mistral": "mistral",
}
_prompt_dir   = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "prompt_services")

missing_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = _SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        fpath = os.path.join(_prompt_dir, f"{file_key}_{variant}_translations.csv")
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if trans_col not in raw_vdf.columns or rat_col not in raw_vdf.columns:
            continue
        has_trans = raw_vdf[trans_col].notna() & ~raw_vdf[trans_col].astype(str).str.strip().isin(["", "nan"])
        has_rat   = raw_vdf[rat_col].apply(_has_rat)
        mismatch  = (has_trans & ~has_rat) | (~has_trans & has_rat)
        for lc in raw_vdf[mismatch]["language_code"].unique():
            missing_rat_by_lang[lc].add(service)

# ── 2. Mixed script, romanization, and placeholder refusals ──────────────────
_ALL_SVC_COLS = {**BASELINE_SERVICES, **LLM_SERVICES}
_baseline_df  = df_raw[df_raw["prompt_variant"] == "minimal"].copy()

mixed_by_lang       = defaultdict(set)
roman_by_lang       = defaultdict(set)
placeholder_by_lang = defaultdict(set)  # model explicitly refused to translate

for service, col in _ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = _baseline_df if service in BASELINE_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        _, action = curate_translation(val)
        lc = row["language_code"]
        if action == "nulled":
            mixed_by_lang[lc].add(service)
        elif action == "stripped":
            roman_by_lang[lc].add(service)
        elif action == "placeholder":
            placeholder_by_lang[lc].add(service)

# ── 3. Script disagreement (from §1.8.1 outlier_df) ──────────────────────────
disagr_by_lang = defaultdict(set)
if not outlier_df.empty:
    for _, row in outlier_df.iterrows():
        disagr_by_lang[row["language_code"]].add(row["service"])

# ── 4. Source term in translation (untranslated leakage) ─────────────────────
# Flags any non-English translation that still contains the English source term
# ("Digital Humanities") or its initials abbreviation ("DH") as a standalone token.
source_term_by_lang = defaultdict(set)
for service, col in _ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = _baseline_df if service in BASELINE_SERVICES else df_raw
    for _, row in src.iterrows():
        lc = row["language_code"]
        if lc == "en":
            continue
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        if has_source_leakage(val, term):
            source_term_by_lang[lc].add(service)

# ── 5. Repetition loops, extreme term length, unicode escapes ────────────────
repeat_loop_by_lang = defaultdict(set)
extreme_len_by_lang = defaultdict(set)
unicode_esc_by_lang = defaultdict(set)
short_by_lang       = defaultdict(set)

for service, col in _ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = _baseline_df if service in BASELINE_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        lc = row["language_code"]
        if is_repetition_loop(val):
            repeat_loop_by_lang[lc].add(service)
        if has_extreme_term_length(val):
            extreme_len_by_lang[lc].add(service)
        if has_unicode_escape(val):
            unicode_esc_by_lang[lc].add(service)
        if has_short_translation(val):
            short_by_lang[lc].add(service)

# ── 6. Any script mixing (including below exclusion threshold) ──────────────
# any_mixing fires whenever secondary-script chars are present at all.
# This is a superset of has_mixed_script (nulled) + has_romanization (stripped):
# it also captures sub-threshold cases that curate_translation leaves unchanged.
any_mixing_by_lang = defaultdict(set)

for service, col in _ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = _baseline_df if service in BASELINE_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        detail = script_mix_detail(val)
        if detail.get("any_mixing"):
            any_mixing_by_lang[row["language_code"]].add(service)

# ── Build DataFrame ───────────────────────────────────────────────────────────
lang_meta = (
    df_raw[["language_code", "language_name"]]
    .drop_duplicates("language_code")
    .dropna(subset=["language_code"])
)
lang_meta = lang_meta.copy()
lang_meta["language_family"] = lang_meta["language_code"].apply(get_language_family)

# Fix any language_name values that fell back to the language code
_ref_path = os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv")
if os.path.exists(_ref_path):
    _ref = pd.read_csv(_ref_path, dtype=str).set_index("language_code")["language_name"]
    _broken = lang_meta["language_name"] == lang_meta["language_code"]
    if _broken.any():
        lang_meta.loc[_broken, "language_name"] = (
            lang_meta.loc[_broken, "language_code"].map(_ref)
            .fillna(lang_meta.loc[_broken, "language_code"])
        )

flag_rows = []
for _, meta in lang_meta.iterrows():
    lc = meta["language_code"]
    m_svcs = sorted(missing_rat_by_lang.get(lc, set()))
    x_svcs = sorted(mixed_by_lang.get(lc, set()))
    r_svcs = sorted(roman_by_lang.get(lc, set()))
    d_svcs = sorted(disagr_by_lang.get(lc, set()))
    l_svcs = sorted(source_term_by_lang.get(lc, set()))
    p_svcs = sorted(placeholder_by_lang.get(lc, set()))
    rl_svcs = sorted(repeat_loop_by_lang.get(lc, set()))
    el_svcs = sorted(extreme_len_by_lang.get(lc, set()))
    ue_svcs = sorted(unicode_esc_by_lang.get(lc, set()))
    sh_svcs = sorted(short_by_lang.get(lc, set()))
    am_svcs = sorted(any_mixing_by_lang.get(lc, set()))
    active = (
        (["missing_rationale"]   if m_svcs  else []) +
        (["mixed_script"]        if x_svcs  else []) +
        (["romanization"]        if r_svcs  else []) +
        (["script_disagreement"] if d_svcs  else []) +
        (["source_term"]         if l_svcs  else []) +
        (["placeholder_term"]    if p_svcs  else []) +
        (["repetition_loop"]     if rl_svcs else []) +
        (["extreme_term_length"] if el_svcs else []) +
        (["unicode_escape"]      if ue_svcs else []) +
        (["short_translation"]   if sh_svcs else []) +
        (["any_mixing"]          if am_svcs else [])
    )
    flag_rows.append({
        "language_code":               lc,
        "language_name":               meta["language_name"],
        "language_family":             meta["language_family"],
        "has_missing_rationale":       bool(m_svcs),
        "missing_rationale_services":  ";".join(m_svcs),
        "has_mixed_script":            bool(x_svcs),
        "mixed_script_services":       ";".join(x_svcs),
        "has_romanization":            bool(r_svcs),
        "romanization_services":       ";".join(r_svcs),
        "has_script_disagreement":     bool(d_svcs),
        "script_disagr_services":      ";".join(d_svcs),
        "has_source_term":             bool(l_svcs),
        "source_term_services":        ";".join(l_svcs),
        "has_placeholder_term":        bool(p_svcs),
        "placeholder_term_services":   ";".join(p_svcs),
        "has_repetition_loop":         bool(rl_svcs),
        "repetition_loop_services":    ";".join(rl_svcs),
        "has_extreme_term_length":     bool(el_svcs),
        "extreme_term_length_services":";".join(el_svcs),
        "has_unicode_escape":          bool(ue_svcs),
        "unicode_escape_services":     ";".join(ue_svcs),
        "has_short_translation":        bool(sh_svcs),
        "short_translation_services":  ";".join(sh_svcs),
        "has_any_mixing":               bool(am_svcs),
        "any_mixing_services":          ";".join(am_svcs),
        "quality_flags":               ";".join(active),
        "flag_count":                  len(active),
    })

quality_flags_df = (
    pd.DataFrame(flag_rows)
    .sort_values(["flag_count", "language_name"], ascending=[False, True])
    .reset_index(drop=True)
)

# ── Save ──────────────────────────────────────────────────────────────────────
flags_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
os.makedirs(flags_dir, exist_ok=True)
flags_path = os.path.join(flags_dir, "quality_flags.csv")
quality_flags_df.to_csv(flags_path, index=False)

flagged = quality_flags_df[quality_flags_df["flag_count"] > 0]
print(f"Saved: {flags_path}")
print(f"  Total languages  : {len(quality_flags_df)}")
print(f"  Flagged          : {len(flagged)} ({len(flagged)/len(quality_flags_df)*100:.1f}%)")
print()
for col, label in [
    ("has_missing_rationale", "Missing rationale "),
    ("has_mixed_script", "Mixed script above threshold"),
    ("has_romanization", "Romanization"),
    ("has_script_disagreement", "Script disagreement"),
    ("has_source_term", "Source term"),
    ("has_placeholder_term", "Placeholder term"),
    ("has_repetition_loop", "Repetition loop"),
    ("has_extreme_term_length", "Extreme length "),
    ("has_unicode_escape", "Unicode escape"),
    ("has_short_translation", "Short translation (<4 codepoints)"),
    ("has_any_mixing",   "Any mixing of scripts (including sub-threshold)"),
]:
    n   = int(quality_flags_df[col].sum())
    pct = n / len(quality_flags_df) * 100
    print(f"  {label}: {n:>3d}  ({pct:.1f}%)")

print()
print(quality_flags_df.head(10)[["language_name", "quality_flags", "flag_count"]].to_string(index=False))


Saved: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/quality_flags.csv
  Total languages  : 880
  Flagged          : 698 (79.3%)

  Missing rationale : 368  (41.8%)
  Mixed script above threshold:  84  (9.5%)
  Romanization: 146  (16.6%)
  Script disagreement: 378  (43.0%)
  Source term: 347  (39.4%)
  Placeholder term:  27  (3.1%)
  Repetition loop:  15  (1.7%)
  Extreme length :  21  (2.4%)
  Unicode escape:   7  (0.8%)
  Short translation (<4 codepoints):  22  (2.5%)
  Any mixing of scripts (including sub-threshold): 256  (29.1%)

     language_name                                                                                        quality_flags  flag_count
       Blissymbols      missing_rationale;romanization;script_disagreement;repetition_loop;short_translation;any_mixing           6
              Boro               missing_rationale;mixed_script;romanization;script_disagreement;source_term;any_mixing  

### Error Categories × Quality Flags

The quality-flags file captures per-language output anomalies (e.g. script mixing, source-term leakage, repetition loops); the error logs capture per-request service failures. Cross-tabulating these two dimensions reveals which pipeline failures tend to co-occur and highlights the most diagnostic language profiles.

Two patterns of particular interest:

**Honest-no doubles** — a language where one service returns a placeholder or undeciphered-script flag while another service honestly refuses (`extinct_ancient` or `knowledge_gap`). These are the most defensible exclusion decisions: multiple independent signals agree the translation is unreliable. Five such languages appear in the data (Meroitic, Ancient North Arabian, Linear A, Elamite, Northern Tutchone).

**Fabricate-vs-passthrough** — a language where one service produces a repetition-loop hallucination while another simply passes through the English source term. Both are failures of different kinds; neither is a usable translation. Ten languages show this split (e.g. `xmr` Meroitic: Gemma loops, Claude/Mistral passthrough; `cch` Atsam: Gemini loops, OpenAI/Claude/DeepSeek/Qwen passthrough).


In [22]:
# Error categories × quality flags cross-tab
_qf = pd.read_csv(
    os.path.join(DATA_DIR, "translated_terms", TARGET_TERMS[0].lower().replace(" ", "_"),
                 "evaluation", "quality_flags.csv"),
    converters={"language_code": str},
)
_flag_cols = [c for c in _qf.columns if c.startswith("has_")]
_err_cats  = ["max_tokens", "api_error", "empty_translation", "ollama_timeout",
              "extinct_ancient", "knowledge_gap", "generic_refusal",
              "repetition_loop", "parse_format", "other"]

# Per-language error-category set
_lang_errcats = (
    llm_err.groupby("language_code")["category"]
    .apply(set)
    .reset_index()
    .rename(columns={"category": "err_cats"})
)
_cross = _qf.merge(_lang_errcats, on="language_code", how="left")
_cross["err_cats"] = _cross["err_cats"].apply(lambda x: x if isinstance(x, set) else set())

_rows = []
for flag in _flag_cols:
    flagged = _cross[_cross[flag] == True]
    for ecat in _err_cats:
        n = flagged["err_cats"].apply(lambda s: ecat in s).sum()
        _rows.append({"flag": flag.replace("has_",""), "error_category": ecat, "n": int(n)})

_xtab = pd.DataFrame(_rows)
xtab_wide = _xtab.pivot(index="flag", columns="error_category", values="n").fillna(0).astype(int)
print("Quality flags × error categories (count = languages with both signals)")
print(xtab_wide[_err_cats].to_string())

# Visualise as a heatmap
xtab_long = _xtab.copy()
xtab_hm = alt.Chart(xtab_long[xtab_long["n"] > 0]).mark_rect().encode(
    x=alt.X("error_category:N", sort=_err_cats, title=None,
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("flag:N", title=None),
    color=alt.Color("n:Q", scale=alt.Scale(scheme="oranges"), title="languages"),
    tooltip=["flag:N", "error_category:N", "n:Q"],
).properties(width=480, height=260, title="Quality flags × error categories")
display(xtab_hm)

# ── Honest-no doubles ────────────────────────────────────────────────────────
print("\nHonest-no doubles (placeholder flag + honest refusal from different service):")
_ph_svc = _qf[_qf["has_placeholder_term"] == True][["language_code","language_name","placeholder_term_services"]]
for _, row in _ph_svc.iterrows():
    lang = row["language_code"]
    ph_svcs = set(str(row["placeholder_term_services"]).split(";"))
    honest = llm_err[(llm_err["language_code"] == lang) &
                     (llm_err["category"].isin(["extinct_ancient","knowledge_gap"])) &
                     (~llm_err["service"].isin(ph_svcs))]
    if not honest.empty:
        print(f"  {lang} ({row['language_name']}): placeholder={sorted(ph_svcs)}, "
              f"honest-refusal={sorted(honest['service'].unique())} ({sorted(honest['category'].unique())})")

# ── Fabricate-vs-passthrough ─────────────────────────────────────────────────
print("\nFabricate-vs-passthrough (repetition-loop + source-term leakage, different services):")
_both = _qf[_qf["has_repetition_loop"] & _qf["has_source_term"]]
for _, r in _both.iterrows():
    rep_svcs = set(str(r["repetition_loop_services"]).split(";"))
    src_svcs = set(str(r["source_term_services"]).split(";"))
    if rep_svcs & src_svcs:   # same service — both flags on same service, less interesting
        diff_src = src_svcs - rep_svcs
        if not diff_src:
            continue
    print(f"  {r['language_code']} ({r['language_name']}): "
          f"loops={sorted(rep_svcs)}, passthrough={sorted(src_svcs)}")


Quality flags × error categories (count = languages with both signals)
error_category       max_tokens  api_error  empty_translation  ollama_timeout  extinct_ancient  knowledge_gap  generic_refusal  repetition_loop  parse_format  other
flag                                                                                                                                                                
any_mixing                    8          7                 13               0                9             37               57               23             2     27
extreme_term_length           2          0                  1               0                2              6                7                3             0      4
missing_rationale            44         14                 11               0               10             90              134               57             2     54
mixed_script                  1          3                  2               0                3          

alt.Chart(...)


Honest-no doubles (placeholder flag + honest refusal from different service):
  rob (Tae'): placeholder=['Claude'], honest-refusal=['OpenAI'] (['knowledge_gap'])
  egy (Ancient Egyptian): placeholder=['Gemini'], honest-refusal=['Llama'] (['extinct_ancient'])
  elx (Elamite): placeholder=['Claude', 'OpenAI'], honest-refusal=['Mistral'] (['extinct_ancient'])
  ecy (Eteocypriot): placeholder=['Claude'], honest-refusal=['OpenAI'] (['knowledge_gap'])
  kro (Kru languages): placeholder=['Claude'], honest-refusal=['OpenAI'] (['knowledge_gap'])
  kut (Kutenai): placeholder=['Claude', 'Mistral'], honest-refusal=['OpenAI'] (['knowledge_gap'])
  lab (Linear A): placeholder=['Claude'], honest-refusal=['OpenAI'] (['knowledge_gap'])
  aro (Araona): placeholder=['Claude', 'DeepSeek', 'Gemini'], honest-refusal=['OpenAI'] (['knowledge_gap'])
  clc (Chilcotin): placeholder=['OpenAI'], honest-refusal=['Llama'] (['knowledge_gap'])
  mvy (Indus Kohistani): placeholder=['Mistral'], honest-refusal=['OpenAI'

## 2.4 Manual Review & Exclusion Summary

The automated quality flags in §2.3 surface *potential* problems; human review in the HTML explorer produces the authoritative exclusion decisions saved in `manual_exclusions.csv`.

Three exclusion types shape all downstream analysis:

- **`analysis_exclusion`** — translation is structurally unusable (loops, untranslatable scripts, pure error tokens). These languages are **dropped entirely** from notebooks 03–08.
- **`search_exclusion`** — translation is analytically interesting but unsafe as a GitHub search string (over-short terms, Braille, extreme repetition). Dropped only in notebook 08.
- **`term_correction`** — minor fixable error (spurious punctuation, source-term wrapper). Corrected form is used in string comparisons.

This section quantifies each type, cross-tabs them against the automated flags to measure auto-pipeline miss rate, and characterises patterns in manual term edits.

In [23]:
from scripts.utils import load_manual_exclusions

term = TARGET_TERMS[0]
_eval_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
analysis_langs, search_terms, corrections = load_manual_exclusions(_eval_dir)

_excl_df = pd.read_csv(os.path.join(_eval_dir, "manual_exclusions.csv"), dtype=str).fillna("")

print("=== Manual Exclusion Counts ===")
print(f"  analysis_exclusion : {len(analysis_langs):>4} unique language codes")
print(f"  search_exclusion   : {len(search_terms):>4} (language, term) pairs  "
      f"({_excl_df['service'].eq('search_exclusion').sum()} rows, "
      f"{_excl_df[_excl_df['service']=='search_exclusion']['language_code'].nunique()} unique langs)")
print(f"  term_correction    : {len(corrections):>4} corrections")
print()

# Overlap: languages that appear in both analysis AND search exclusions
_ae_langs = set(_excl_df[_excl_df["service"]=="analysis_exclusion"]["language_code"])
_se_langs = set(_excl_df[_excl_df["service"]=="search_exclusion"]["language_code"])
print(f"  Overlap (both analysis + search exclusion): {len(_ae_langs & _se_langs)} languages")
print(f"  Total excluded from ≥1 analysis role      : {len(_ae_langs | _se_langs)} languages")
print(f"  Unaffected languages                       : {len(_qf["language_code"].unique()) - len(_ae_langs | _se_langs)}")

=== Manual Exclusion Counts ===
  analysis_exclusion :   84 unique language codes
  search_exclusion   :  209 (language, term) pairs  (209 rows, 131 unique langs)
  term_correction    :   82 corrections

  Overlap (both analysis + search exclusion): 24 languages
  Total excluded from ≥1 analysis role      : 192 languages
  Unaffected languages                       : 688


### Auto-Pipeline Miss Rate

For each language that received a manual `analysis_exclusion`, we check whether the automated quality-flag pipeline would have caught it:

- **Caught**: the language had at least one quality flag (`has_repetition_loop`, `has_placeholder_term`, `has_unicode_escape`, `has_extreme_term_length`, `has_short_translation`).
- **Missed**: the language had *zero* quality flags — the auto-pipeline saw nothing suspicious.

A high miss rate implies that manual review remains essential even after automated filtering.

In [24]:
_qf = pd.read_csv(os.path.join(_eval_dir, "quality_flags.csv"), converters={"language_code": str})

_AUTO_CATCH_FLAGS = [
    "has_repetition_loop", "has_placeholder_term", "has_unicode_escape",
    "has_extreme_term_length", "has_short_translation",
]
_flag_cols_present = [c for c in _AUTO_CATCH_FLAGS if c in _qf.columns]

_ae_qf = _qf[_qf["language_code"].isin(analysis_langs)].copy()
_ae_qf["any_auto_flag"] = _ae_qf[_flag_cols_present].apply(
    lambda row: any(str(v).strip().lower() == "true" for v in row), axis=1
)
_caught = _ae_qf["any_auto_flag"].sum()
_missed = len(_ae_qf) - _caught
_miss_rate = _missed / len(_ae_qf) if len(_ae_qf) else 0

print(f"Analysis-excluded languages: {len(_ae_qf)}")
print(f"  Caught by auto-flags : {_caught}  ({_caught/len(_ae_qf):.0%})")
print(f"  Missed by auto-flags : {_missed}  ({_miss_rate:.0%})")
print()
print("Missed languages (zero quality flags):")
_missed_df = _ae_qf[~_ae_qf["any_auto_flag"]][["language_code", "language_name"] + _flag_cols_present].reset_index(drop=True)
print(_missed_df.to_string(index=False))
print()

# Which auto-flag types most often co-occur with analysis exclusions?
print("Auto-flag co-occurrence with analysis_exclusion (among caught):")
for col in _flag_cols_present:
    n = _ae_qf[col].apply(lambda v: str(v).strip().lower() == "true").sum()
    print(f"  {col:<35} {n:>3}  ({n/len(_ae_qf):.0%})")

Analysis-excluded languages: 84
  Caught by auto-flags : 57  (68%)
  Missed by auto-flags : 27  (32%)

Missed languages (zero quality flags):
language_code         language_name  has_repetition_loop  has_placeholder_term  has_unicode_escape  has_extreme_term_length  has_short_translation
          shn                  Shan                False                 False               False                    False                  False
          xna Ancient North Arabian                False                 False               False                    False                  False
          xlc                Lycian                False                 False               False                    False                  False
          lwl          Eastern Lawa                False                 False               False                    False                  False
          kfo                  Koro                False                 False               False                    False

### Term Correction Patterns

Each `term_correction` row records a manual edit to a translation. Categorising these edits reveals what kinds of artefacts the LLMs most consistently introduce.

In [25]:
import re

_corr_df = _excl_df[_excl_df["service"] == "term_correction"].copy()
_corr_df = _corr_df[_corr_df["corrected_term"].str.strip() != ""].reset_index(drop=True)

def _classify_edit(orig, corr):
    orig, corr = str(orig).strip(), str(corr).strip()
    src_term = term  # "Digital Humanities"
    # Leading / trailing punctuation stripped
    if corr == re.sub(r'^[\s\W]+|[\s\W]+$', '', orig):
        return "strip_punctuation"
    # Source term wrapper removed  (e.g. "Digital Humanities (X)" → "X")
    if src_term in orig and src_term not in corr:
        return "remove_source_wrapper"
    # Leading slash removed  (e.g. "/Dkotan" → "Dkotan")
    if orig.startswith("/") and corr == orig[1:]:
        return "remove_leading_slash"
    # Parenthetical stripped from end
    if re.search(r'\(.+\)$', orig) and corr == re.sub(r'\s*\(.+\)$', '', orig).strip():
        return "strip_trailing_parenthetical"
    # Transliteration / script cleanup (corrected has different script)
    return "other"

_corr_df["edit_type"] = _corr_df.apply(
    lambda r: _classify_edit(r["original_term"], r["corrected_term"]), axis=1
)

print("Term correction edit types:")
print(_corr_df["edit_type"].value_counts().to_string())
print(f"\nTotal corrections: {len(_corr_df)}")
print()
print("Examples per edit type:")
for etype, grp in _corr_df.groupby("edit_type"):
    ex = grp[["language_code", "original_term", "corrected_term"]].head(3)
    print(f"\n  [{etype}]")
    for _, r in ex.iterrows():
        print(f"    {r['language_code']}: {repr(r['original_term'][:60])} → {repr(r['corrected_term'][:60])}")

Term correction edit types:
edit_type
strip_trailing_parenthetical    41
remove_source_wrapper           17
strip_punctuation               15
other                            7
remove_leading_slash             2

Total corrections: 82

Examples per edit type:

  [other]
    saq: 'Kiswahili: Uchaguzi wa Maarifa Digitaali' → 'Uchaguzi wa Maarifa Digitaali'
    cr: 'ᒪᒪᐗᐤ ᑭᐢᑫᔨᐦᑕᒧᐎᓇ / mamawâw kiskêyihtamowin' → 'ᒪᒪᐗᐤ ᑭᐢᑫᔨᐦᑕᒧᐎᓇ'
    crk: 'ᑎᒋᑕᓪ ᐊᔭᒥᐦᐃᑐᐏᓇ / dijital ayamihitowin' → 'ᑎᒋᑕᓪ ᐊᔭᒥᐦᐃᑐᐏᓇ'

  [remove_leading_slash]
    gld: '/Dkaw ᑕ+-+-+' → 'Dkaw ᑕ+-+-+'
    hnn: "/Dkawg'" → "Dkawg'"

  [remove_source_wrapper]
    brx: 'Digital Humanities (Dhigital Huministis)' → 'Dhigital Huministis'
    brx: 'Digital Humanities ≈ ཐོགས་པའི་བདེན་ཆུང' → 'ཐོགས་པའི་བདེན་ཆུང'
    lus: 'Digital Humanities (Dhual Hlai Mual)' → 'Dhual Hlai Mual'

  [strip_punctuation]
    ain: '/Dkotan Hantekitika' → 'Dkotan Hantekitika'
    ain: '/Dkara Usaamrra' → 'Dkara Usaamrra'
    lus: '/Dkawm Hmuantun' → 'Dkawm Hmuantun